# SIRAP USA — Backend: RT-DETRv2-S, OCR, and OpenVINO

**SIRAP — Automatic License Plate Detection and Recognition System** · USA research and preparation notebook

**Purpose.** This notebook prepares and evaluates SIRAP's USA profile. It connects dataset preparation, RT-DETRv2-S training, OpenVINO deployment, OCR, complete-scene evaluation, and optional camera and vehicle-spec lookup workflows. The notebook is a research and deployment-preparation workflow; running a cell produces results only when its required data and run switch are enabled.

**Execution switches.** Training, threshold calibration, robustness/latency/OCR, end-to-end evaluations, the final test, J.D. Power lookup, metric publishing, live-camera processing, and Space publication are disabled by default. Enable one stage at a time only after reviewing its inputs and cost.

**Current checkpoint status.** Seed 1 was selected by grouped Roboflow validation (mAP50-95 = 0.8321). The manual full-scene validation has 71 images: 41 with plates and 30 PKLot negatives. Its best observed candidate (confidence 0.60) achieved precision 0.9302 (40 true positives, 3 false positives, 1 false negative), below the 0.99 target. The saved report therefore has no operational threshold and blocks inference. Do not use this checkpoint for access decisions. An existing independent-test JSON records a different seed-0 selection and is not evidence for the current seed-1 checkpoint; the selected checkpoint's independent test remains unrun.

**Recognition flow.** For an image or video frame, the detector proposes license-plate boxes. SIRAP crops each candidate and reads two distinct fields: the printed issuing state and the registration serial. State-name OCR uses Tesseract and maps recognized names or clear two-letter codes to a state abbreviation. Serial OCR uses the global plate OCR model. Together, the two fields form a jurisdiction-qualified key such as \`CA:6ZGY984\`. Section 23 can optionally use a plate/state pair to request vehicle-level report details.

1. **Detect:** RT-DETRv2-S proposes plate boxes.
2. **Read:** Tesseract scans the state-name field; serial OCR reads the registration characters.
3. **Combine:** SIRAP records the state, abbreviation, serial, confidence, and lookup key separately.
4. **Evaluate:** detector, crop OCR, full-scene, pipeline, robustness, and timing checks report distinct outcomes.
5. **Deploy or inspect:** optional camera sections run the model locally or through the Cloudflare bridge; optional publication sends the reviewed detector IR to the Space.

**Detector and runtime.** Fine-tuning uses Transformers and PyTorch from the Apache-2.0-declared starting checkpoint [merve/license-plates-rtdetrv2](https://huggingface.co/merve/license-plates-rtdetrv2). The trained detector is exported to OpenVINO IR and runs through native Python OpenVINO Runtime. Dataset and dependency licenses have their own terms and must be reviewed separately.

**Serial OCR runtime.** The OCR model artifact is ONNX. The dependency setup detects the CPU vendor: Intel CPUs use ONNX Runtime with OpenVINO Execution Provider prioritized and CPU provider fallback; other CPUs use ONNX Runtime's CPU provider. This selection applies to serial OCR. The RT-DETR detector continues to use its OpenVINO IR path.

**Storage layout.** GitHub holds the notebook, code, configurations, and compact aggregate metrics. Section 2 mounts Google Drive: dataset archives, selected training checkpoints, large models, and videos are kept there. Row-level OCR results, camera readings, and vehicle lookups go to the private Drive folder `SIRAP_USA_ECUADOR/private/usa`. Training and evaluation use local `/content` copies for speed, then persist selected outputs to Drive. Add `GITHUB_TOKEN` to Colab Secrets to publish compact metrics; `ROBOFLOW_API_KEY` is needed only for a fresh Roboflow download. Keep the Drive private folder unshared.

**Uncertainty intervals.** Sections 20 and 21 report 95% Wilson score intervals for exact-match and missed-detection proportions, plus 95% percentile-bootstrap intervals for exact-match rate and character error rate. The bootstrap resamples labeled scene images as paired units (5,000 resamples). These intervals quantify sampling uncertainty for the supplied scenes; they do not adjust for groups of related images.

## Notebook contents

| Section | What it does |
|---|---|
| 0. Install dependencies | Installs Transformers/PyTorch, OpenVINO, the CPU-selected OCR runtime, Tesseract, Roboflow, and image/data utilities. |
| 1. Clone or refresh the project repository | Prepares a clean project clone and adds it to Python's import path. |
| 2. Set paths, model identity, and compute information | Mounts Drive, defines persistent and local cache paths, and reports the available training hardware. |
| 3. Inspect the USA profile | Reads the country configuration and checks the expected dataset and model folders. |
| 4. Select and download the Roboflow dataset | Selects the European + USA License Plates source or a manually supplied COCO dataset. |
| 5. Convert full-scene annotations to COCO | Provides an optional route from reviewed scene boxes and a CSV template to COCO. |
| 6. Create the grouped holdout | Audits hashes and source groups, prevents split leakage, and builds a larger independent test split. |
| 7. Validate COCO splits | Checks the generated annotations, paths, counts, and split structure. |
| 8. Visually inspect annotations | Displays representative training images and bounding boxes. |
| 9. Recommended training protocol | Explains the training and model-selection approach before compute-heavy runs. |
| 10. Set training, search, and seed parameters | Defines training duration, patience, search, batch, and reproducibility settings. |
| 11. Optional Optuna search | Searches selected training hyperparameters on train/validation data. |
| 12. Train and select the three-seed candidate | Trains candidate runs, compares validation results, and records the selected checkpoint. |
| 13. Load the USA OpenVINO IR | Loads the selected detector IR for inference. |
| 14. Run an OpenVINO smoke check | Confirms the exported detector can perform a basic inference pass. |
| 15. Calibrate the confidence threshold | Selects the operating threshold from validation data and records its checkpoint identity. |
| 16. Optional live-camera mode | Connects the USA pipeline to a camera through a Cloudflare bridge or runs it beside the camera. Subsections 16.1–16.5 create the bridge, configure the reader, check connectivity, run a bounded session, and generate the local edge runner. |
| 17. Image robustness checks | Measures detector behavior under documented image perturbations. |
| 18. CPU inference latency | Measures detector timing under the recorded hardware and input conditions. |
| 19. Read and evaluate the two USA plate fields | Reads state name/code and registration serial separately, compares them with available labels, and writes field-level diagnostics. |
| 20. Full-scene end-to-end evaluation | Evaluates detector plus OCR on a separate labeled scene set. |
| 21. SIRAP pipeline end-to-end evaluation | Runs the complete application pipeline on a separate labeled scene set and reports Wilson and bootstrap confidence intervals. |
| 22. One-time held-out detector test | Produces the final detector test report after checkpoint selection and threshold calibration; do not use it to revise the model. |
| 23. Optional J.D. Power lookup | Installs Playwright 1.40.0 and Chromium, then looks up vehicle-level details for the supplied sample pairs or a capped set from section 19. Its output is separate from model metrics. |
| 24. Pre-deployment checklist | Reviews data coverage, split integrity, metric scope, threshold selection, and operational safeguards. |
| 25. Optional OpenVINO IR publication | Publishes the reviewed detector IR to the Hugging Face Space when explicitly enabled. |
| 26. Result files and reporting | Lists generated reports and explains which split and settings must accompany reported metrics. |

## How to use the notebook

Open this notebook in Colab and run cells in order, starting at section 0. If Colab requests a runtime restart after installing dependencies, restart and continue from the repository setup; section 2 will ask you to mount Drive. Training and model selection use train/validation data. Freeze the selected checkpoint and calibrate its threshold before running the final detector test in section 22, which is designed to run once.

| To run | Run first |
|---|---|
| Dataset download and grouped COCO preparation | Sections 0–8; section 2 mounts Drive, then section 4 restores a cached dataset or downloads and archives it before local split preparation. |
| Training and candidate selection | Sections 0–10, then 11 if using Optuna, followed by section 12. Training runs in `/content`; the selected checkpoint is copied to Drive. |
| OpenVINO inference and threshold calibration | Sections 0–15 in order, with the selected checkpoint available. |
| Separate state/serial crop OCR | Sections 0–3 and 19; provide the USA crop images and ground-truth CSV. Detailed per-crop rows are stored in private Drive; aggregate metrics go to GitHub. |
| Full-scene or SIRAP pipeline evaluation | Freeze the checkpoint, calibrate the threshold, and prepare the separate labeled scene CSV before section 20 or 21. |
| Live camera | Prepare the model and threshold, then run 16.1–16.4 with the camera bridge and required Colab Secrets. Section 16.5 generates a runner for the camera-side PC. |
| J.D. Power vehicle-spec lookup | Run 23.1, then 23.2. The section-19 CSV option reads the private Drive OCR table; lookup rows are saved to private Drive. |
| Publish the detector IR | Review the final results first, then enable section 25 and provide the required repository secret. |

**Outputs and reporting.** Training and evaluation cells are controlled by run switches and require the referenced data and checkpoints. Small aggregate metrics are committed to `results/metrics/` when `GITHUB_TOKEN` is available. Large or private artifacts are copied to the mounted Drive. Lookup rows are separate vehicle-information results, not detector or OCR metrics. Do not present any metric as a result until the corresponding cell has completed and its dataset, split, checkpoint, threshold, runtime, and settings have been reviewed.

## Workflow and execution order

The first run prepares the environment and repository, mounts Drive in section 2, restores or selects a dataset, and builds an audited COCO split. Dataset archives and the generated independent split are cached on Drive; working copies are extracted to local `/content` for processing. Training and tuning use train/validation data. Training checkpoints stay on local `/content` during compute, then the selected checkpoint is copied to Drive. After selecting and freezing a checkpoint, calibrate its threshold on validation; reserve the grouped test for one final evaluation.

For a remote test, use the Cloudflare bridge with a temporary Quick Tunnel or a named tunnel protected by Access. For on-site use, generate the edge runner in section 16.5 and run OpenVINO inference on the camera PC. The edge runner saves minimal event rows locally. Its optional network upload stays disabled because this repository does not yet expose a dedicated USA machine-event route. Both modes keep video frames out of the SIRAP event payload.

## 0. Install dependencies

This cell installs the Transformers/PyTorch training tools, COCO metrics, native OpenVINO Runtime for the detector, Tesseract, Roboflow, YAML, HTTP and image utilities. It detects the CPU vendor and installs fast-plate-ocr with ONNX Runtime plus the OpenVINO Execution Provider on Intel CPUs, or ONNX Runtime's standard CPU provider elsewhere. If Colab asks to restart, restart and then continue from the repository setup section.

In [ ]:
import platform
import re
import subprocess
import sys
from pathlib import Path

def detect_intel_cpu():
    """Use the OpenVINO EP package only when the host reports an Intel CPU."""
    details = f"{platform.processor()} {platform.uname().processor}"
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.is_file():
        try:
            details += " " + cpuinfo.read_text(errors="ignore")
        except OSError:
            pass
    has_intel = bool(re.search(r"genuineintel|intel\(r\)|\bintel\b", details, re.I))
    has_amd = bool(re.search(r"authenticamd|\bamd\b", details, re.I))
    return has_intel and not has_amd

INTEL_CPU_DETECTED = detect_intel_cpu()
OCR_PACKAGE = (
    "fast-plate-ocr[onnx-openvino]==1.1.0"
    if INTEL_CPU_DETECTED
    else "fast-plate-ocr[onnx]==1.1.0"
)
packages = [
    "transformers>=5.4,<6",
    "accelerate>=1.2,<2",
    "pycocotools>=2.0.8",
    "optuna>=4,<5",
    "openvino>=2025.4,<2027",
    OCR_PACKAGE,
    "pytesseract",
    "roboflow==1.7.1",
    "opencv-python-headless==4.12.0.88",
    "matplotlib>=3.8",
    "pandas>=2.2,<4",
    "PyYAML>=6,<7",
    "requests>=2.31,<3",
    "tqdm>=4.66,<5",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("OCR backend package:", OCR_PACKAGE)
print("Intel CPU detected:", INTEL_CPU_DETECTED)
print("Dependencies installed. If Colab asks to restart the runtime, restart it and then continue from the repository setup section.")

## 1. Clone or refresh the project repository

This cell reuses and fast-forward-updates a clean clone when possible. Otherwise, it makes a fresh clone in one of two Colab locations. It refuses to overwrite non-empty folders or a clone with uncommitted changes, then adds the repository to Python's import path.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git"
preferred = Path("/content/SIRAP_USA_ECUADOR")
candidates = [preferred, Path("/content/SIRAP_USA_ECUADOR_notebook")]

def prepare_clone(path):
    if (path / ".git").is_dir():
        status = subprocess.run(
            ["git", "-C", str(path), "status", "--porcelain"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if status:
            return False
        subprocess.run(
            ["git", "-C", str(path), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
        return True
    if path.exists() and any(path.iterdir()):
        return False
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(path)], check=True)
    return True

REPO_ROOT = next((path for path in candidates if prepare_clone(path)), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "No clean clone folder was available. Existing folders were left untouched. "
        "Check /content/SIRAP_USA_ECUADOR and /content/SIRAP_USA_ECUADOR_notebook."
    )
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPO_ROOT)

## 2. Set paths, model identity, and compute information

This setup mounts your Google Drive and defines the storage layout. Training runs and dataset extraction stay in local `/content` for speed; source and independent COCO splits, the selected Transformers/OpenVINO checkpoint, and large artifacts are archived in Drive. Compact, non-row-level metrics go to the cloned repository's `results/metrics/` and are published with `GITHUB_TOKEN`. Raw OCR, camera, and vehicle-lookup tables go only to `MyDrive/SIRAP_USA_ECUADOR/private/usa`; keep that folder unshared. The suggested batch size is 8 for a GPU with at least 16 GB, otherwise 4. Fine-tuning is intended for a Colab GPU; CPU training can be very slow.

In [ ]:
import json
import math
import shutil
import time
import hashlib
import os
import tarfile
import tempfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path
from google.colab import drive

# Google Drive is persistent storage; compute-heavy reads and writes use local /content.
drive.mount("/content/drive_sirap")
DRIVE_ROOT = Path("/content/drive_sirap/MyDrive/SIRAP_USA_ECUADOR")
DRIVE_DATA_ROOT = DRIVE_ROOT / "datasets/usa"
DRIVE_DATA_ARCHIVE_DIR = DRIVE_DATA_ROOT / "archives"
DRIVE_MANUAL_SOURCE_IMAGES = DRIVE_DATA_ROOT / "manual_scene_source_images"
DRIVE_MANUAL_ANNOTATIONS_CSV = DRIVE_DATA_ROOT / "manual_scene_annotations.csv"
DRIVE_MODEL_ROOT = DRIVE_ROOT / "models/usa/detector/rtdetrv2_s"
DRIVE_SHARED_IR_DIR = DRIVE_ROOT / "models/shared/detector/rtdetrv2_s/openvino"
DRIVE_CHECKPOINTS_ROOT = DRIVE_ROOT / "checkpoints/usa/rtdetrv2_s"
DRIVE_PRIVATE_ROOT = DRIVE_ROOT / "private/usa"
DRIVE_PRIVATE_OCR_DIR = DRIVE_PRIVATE_ROOT / "ocr"
DRIVE_PRIVATE_LOOKUP_DIR = DRIVE_PRIVATE_ROOT / "vehicle_lookup"
DRIVE_PRIVATE_CAMERA_DIR = DRIVE_PRIVATE_ROOT / "camera_sessions"
DRIVE_PRIVATE_EVALUATION_DIR = DRIVE_PRIVATE_ROOT / "evaluation"
DRIVE_SCRIPTS_DIR = DRIVE_ROOT / "generated_scripts"
DRIVE_RESULTS_ROOT = DRIVE_ROOT / "results/usa"

LOCAL_DATA_CACHE_ROOT = Path("/content/SIRAP_workspace/datasets/usa")
LOCAL_MANUAL_DATA_ROOT = LOCAL_DATA_CACHE_ROOT / "manual_scene_coco"
LOCAL_SOURCE_CACHE_PARENT = LOCAL_DATA_CACHE_ROOT / "source_cache"
LOCAL_INDEPENDENT_CACHE_PARENT = LOCAL_DATA_CACHE_ROOT / "independent_cache"
LOCAL_INDEPENDENT_ROOT = LOCAL_INDEPENDENT_CACHE_PARENT / "european-usa-license-plates-independent-coco"
LOCAL_PRIVATE_ROOT = Path("/content/SIRAP_workspace/private/usa")
LOCAL_PRIVATE_OCR_DIR = LOCAL_PRIVATE_ROOT / "ocr"
LOCAL_PRIVATE_LOOKUP_DIR = LOCAL_PRIVATE_ROOT / "vehicle_lookup"
LOCAL_PRIVATE_EVALUATION_DIR = LOCAL_PRIVATE_ROOT / "evaluation"

for folder in (
    DRIVE_DATA_ARCHIVE_DIR, DRIVE_MODEL_ROOT.parent, DRIVE_SHARED_IR_DIR,
    DRIVE_CHECKPOINTS_ROOT, DRIVE_PRIVATE_OCR_DIR, DRIVE_PRIVATE_LOOKUP_DIR,
    DRIVE_PRIVATE_CAMERA_DIR, DRIVE_PRIVATE_EVALUATION_DIR, DRIVE_SCRIPTS_DIR,
    DRIVE_RESULTS_ROOT, LOCAL_DATA_CACHE_ROOT, LOCAL_PRIVATE_OCR_DIR,
    LOCAL_PRIVATE_LOOKUP_DIR, LOCAL_PRIVATE_EVALUATION_DIR,
):
    folder.mkdir(parents=True, exist_ok=True)

def archive_directory_to_drive(source_dir, archive_path, archive_root_name):
    """Archive a completed directory locally, then copy one portable file to Drive."""
    source_dir = Path(source_dir).resolve()
    archive_path = Path(archive_path)
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Directory to archive does not exist: {source_dir}")
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(prefix="sirap_", suffix=".tar", dir="/content")
    os.close(descriptor)
    temporary_archive = Path(temporary_name)
    staged_archive = archive_path.with_name(archive_path.name + ".partial")
    try:
        with tarfile.open(temporary_archive, "w") as archive:
            # Independent split images may be links to source images. Dereference them
            # so the archived split remains self-contained in a later Colab session.
            archive.dereference = True
            archive.add(source_dir, arcname=archive_root_name)
        staged_archive.unlink(missing_ok=True)
        shutil.copy2(temporary_archive, staged_archive)
        staged_archive.replace(archive_path)
    finally:
        temporary_archive.unlink(missing_ok=True)
        staged_archive.unlink(missing_ok=True)
    print("Drive archive saved:", archive_path)
    return archive_path

def restore_directory_archive(archive_path, local_parent, expected_root_name):
    """Restore a Drive archive into local Colab storage and reject unsafe archive paths."""
    archive_path = Path(archive_path)
    local_parent = Path(local_parent)
    if not archive_path.is_file():
        raise FileNotFoundError(f"Drive archive not found: {archive_path}")
    if local_parent.exists():
        shutil.rmtree(local_parent)
    local_parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r") as archive:
        members = archive.getmembers()
        for member in members:
            member_path = Path(member.name)
            if (
                member_path.is_absolute()
                or ".." in member_path.parts
                or member.issym()
                or member.islnk()
            ):
                raise ValueError(f"Unsafe archive entry: {member.name}")
        archive.extractall(local_parent, members=members)
    restored_root = local_parent / expected_root_name
    if not restored_root.is_dir():
        raise FileNotFoundError(f"Expected folder missing from archive: {restored_root}")
    print("Restored to local Colab storage:", restored_root)
    return restored_root

def persist_file_to_drive(source_path, drive_path):
    """Copy a completed artifact from local Colab storage to its persistent Drive location."""
    source_path, drive_path = Path(source_path), Path(drive_path)
    if not source_path.is_file():
        raise FileNotFoundError(f"Local artifact to persist does not exist: {source_path}")
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, drive_path)
    print("Copied artifact to Drive:", drive_path)
    return drive_path

def persist_tree_to_drive(source_dir, drive_dir):
    """Copy a completed output directory to Drive after local processing finishes."""
    source_dir, drive_dir = Path(source_dir), Path(drive_dir)
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Local output directory to persist does not exist: {source_dir}")
    if drive_dir.exists():
        shutil.rmtree(drive_dir)
    drive_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dir, drive_dir, symlinks=False)
    print("Copied output directory to Drive:", drive_dir)
    return drive_dir

PUBLISH_SMALL_METRICS_TO_GITHUB = False
def publish_small_metrics(paths, message):
    """Publish only existing compact, non-row-level reports from the repository clone."""
    if not PUBLISH_SMALL_METRICS_TO_GITHUB:
        return []
    relative_paths = []
    for item in paths:
        path = Path(item)
        if path.is_absolute():
            path = path.resolve().relative_to(REPO_ROOT.resolve())
        target = REPO_ROOT / path
        if target.is_file():
            relative_paths.append(path.as_posix())
    if not relative_paths:
        print("No compact GitHub metrics are ready to publish.")
        return []
    from scripts.notebook_io import publish_artifacts
    return publish_artifacts(REPO_ROOT, relative_paths, message)

COUNTRY = "usa"
MODEL_ID = "merve/license-plates-rtdetrv2"
IMAGE_SIZE = 640
DATA_DIR = None
MODEL_OUTPUT = DRIVE_MODEL_ROOT
REPO_MODEL_OUTPUT = REPO_ROOT / "models/usa/detector/rtdetrv2_s"
METRICS_DIR = REPO_ROOT / "results/metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_ROOT = Path("/content/rtdetrv2_usa_runs")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = torch.cuda.is_available()
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if GPU_AVAILABLE else 0.0
)
BATCH_SIZE = 8 if GPU_MEMORY_GB >= 16 else 4

print("Country:", COUNTRY)
print("Starting checkpoint:", MODEL_ID)
print("Training device:", torch.cuda.get_device_name(0) if GPU_AVAILABLE else "CPU")
print(f"GPU memory: {GPU_MEMORY_GB:.1f} GB | recommended initial batch size: {BATCH_SIZE}")
print("Dataset root: not selected yet")

# The detector uses native OpenVINO IR. OCR keeps the ONNX artifact and selects
# OpenVINO Execution Provider on Intel CPUs, or ONNX Runtime CPU elsewhere.
import platform
import re
from types import SimpleNamespace

import cv2
import onnxruntime as ort
import requests
import yaml
from fast_plate_ocr import LicensePlateRecognizer

OCR_MODEL_NAME = "cct-s-v2-global-model"

def detect_intel_cpu():
    """Detect Intel from CPU-identification strings, not from the GPU or OS name."""
    details = f"{platform.processor()} {platform.uname().processor}"
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.is_file():
        try:
            details += " " + cpuinfo.read_text(errors="ignore")
        except OSError:
            pass
    has_intel = bool(re.search(r"genuineintel|intel\(r\)|\bintel\b", details, re.I))
    has_amd = bool(re.search(r"authenticamd|\bamd\b", details, re.I))
    return has_intel and not has_amd

INTEL_CPU_DETECTED = detect_intel_cpu()

def create_serial_ocr_model(model_name=OCR_MODEL_NAME):
    """Create the ORT OCR session with the provider required for this CPU vendor."""
    available = ort.get_available_providers()
    if INTEL_CPU_DETECTED:
        if "OpenVINOExecutionProvider" not in available:
            raise RuntimeError(
                "An Intel CPU was detected, but OpenVINOExecutionProvider is unavailable. "
                "Rerun the dependency cell so it installs fast-plate-ocr[onnx-openvino]."
            )
        providers = ["OpenVINOExecutionProvider"]
        if "CPUExecutionProvider" in available:
            providers.append("CPUExecutionProvider")
    else:
        if "CPUExecutionProvider" not in available:
            raise RuntimeError(
                "The non-Intel OCR route requires ONNX Runtime's CPUExecutionProvider."
            )
        providers = ["CPUExecutionProvider"]

    model = LicensePlateRecognizer(model_name, providers=providers)
    active = model.model.get_providers()
    if INTEL_CPU_DETECTED and (
        not active or active[0] != "OpenVINOExecutionProvider"
    ):
        raise RuntimeError(
            "The Intel OCR session did not register OpenVINOExecutionProvider first. "
            f"Registered providers: {active}"
        )
    if not INTEL_CPU_DETECTED and (
        not active or active[0] != "CPUExecutionProvider"
    ):
        raise RuntimeError(
            f"The non-Intel OCR session must use ONNX Runtime CPU; registered: {active}"
        )
    return model, active

class RuntimeSelectedSerialReader:
    """ONNX OCR reader with an Intel OpenVINO EP / non-Intel ORT CPU selection."""

    def __init__(self, model_name=OCR_MODEL_NAME):
        self.model, self.providers = create_serial_ocr_model(model_name)
        self.backend = (
            "ONNX Runtime + OpenVINOExecutionProvider"
            if INTEL_CPU_DETECTED
            else "ONNX Runtime + CPUExecutionProvider"
        )
        self.runtime_name = self.backend

    def run(self, *args, **kwargs):
        return self.model.run(*args, **kwargs)

    def read(self, crop):
        from src.data_processing import clean_text

        prediction = self.run(
            np.ascontiguousarray(crop),
            return_confidence=True,
        )[0]
        text = clean_text(prediction.plate)
        probabilities = np.asarray(
            prediction.char_probs,
            dtype=np.float32,
        ).ravel()[:max(len(text), 1)]
        if not probabilities.size:
            return text, 0.0, 0.0
        return text, float(probabilities.min()), float(probabilities.mean())


## 3. Inspect the USA profile and expected directories

This exploratory cell reads the country YAML used by the application and prints the configured detector. It also checks whether USA dataset, evaluation, and model directories exist. Missing directories are informational at this point.

In [ ]:
import yaml
cfg=yaml.safe_load((REPO_ROOT/"config/countries/usa.yaml").read_text())
print(cfg["detector"])
for p in [REPO_ROOT/"datasets/usa",REPO_ROOT/"evaluation/usa",REPO_ROOT/"models/usa"]:
    print(p.relative_to(REPO_ROOT),"exists:",p.exists())


## 4. Select and download the Roboflow dataset

The default source is European + USA License plates, workspace plateblur, project european-usa-license-plates, version 1, exported as COCO Segmentation. RT-DETR uses the COCO boxes and maps the three plate labels to one license_plate class during split creation. The first download is archived in Drive; later runs restore it to local `/content`. Add ROBOFLOW_API_KEY to Colab Secrets only when a fresh download is needed. Set USE_ROBOFLOW to False for the manual full-scene route below. Set REFRESH_DATASET_CACHE to True to download/recache the source again, or REBUILD_INDEPENDENT_SPLIT to True to rebuild its grouped split from the cached source.

An earlier inspection of version 1 reported 13,143 training images, 386 validation images, and 106 test images. The current run reads actual counts from the downloaded annotations; these figures are dataset counts, not model results.

In [ ]:
from src.train import (
    RF_FORMAT,
    RF_PROJECT,
    RF_TITLE,
    RF_VERSION,
    RF_WORKSPACE,
    _annotation_file,
    download_roboflow_dataset,
)

DATA_DIR = None
USE_ROBOFLOW = True
REFRESH_DATASET_CACHE = False
REBUILD_INDEPENDENT_SPLIT = False
DATASET_IS_CACHED_INDEPENDENT_SPLIT = False
SOURCE_DATASET_NAME = RF_TITLE if USE_ROBOFLOW else "Manually supplied USA COCO dataset"
ROBOFLOW_VERSION_USED = RF_VERSION if USE_ROBOFLOW else None

DATASET_CACHE_KEY = (
    "european_usa_license_plates_v1"
    if USE_ROBOFLOW
    else "manual_usa_coco"
)
DRIVE_SOURCE_DATA_ARCHIVE = DRIVE_DATA_ARCHIVE_DIR / f"{DATASET_CACHE_KEY}_source.tar"
DRIVE_INDEPENDENT_DATA_ARCHIVE = DRIVE_DATA_ARCHIVE_DIR / f"{DATASET_CACHE_KEY}_independent.tar"

if USE_ROBOFLOW:
    print(
        f"Roboflow source: {RF_TITLE} | "
        f"{RF_WORKSPACE}/{RF_PROJECT} | version {RF_VERSION} | {RF_FORMAT}"
    )
else:
    print("Dataset source: manually supplied USA COCO dataset")

print("Drive dataset archive folder:", DRIVE_DATA_ARCHIVE_DIR)

if (
    not REFRESH_DATASET_CACHE
    and not REBUILD_INDEPENDENT_SPLIT
    and DRIVE_INDEPENDENT_DATA_ARCHIVE.is_file()
):
    DATA_DIR = str(
        restore_directory_archive(
            DRIVE_INDEPENDENT_DATA_ARCHIVE,
            LOCAL_INDEPENDENT_CACHE_PARENT,
            LOCAL_INDEPENDENT_ROOT.name,
        )
    )
    DATASET_IS_CACHED_INDEPENDENT_SPLIT = True
    print("Using the audited independent COCO split cached on Drive.")
elif not REFRESH_DATASET_CACHE and DRIVE_SOURCE_DATA_ARCHIVE.is_file():
    DATA_DIR = str(
        restore_directory_archive(
            DRIVE_SOURCE_DATA_ARCHIVE,
            LOCAL_SOURCE_CACHE_PARENT,
            "dataset",
        )
    )
    print("Using the COCO source dataset cached on Drive.")
elif USE_ROBOFLOW:
    # Add ROBOFLOW_API_KEY to Colab Secrets only when a fresh download is required.
    DATA_DIR = str(download_roboflow_dataset("/content/datasets"))
    archive_directory_to_drive(
        DATA_DIR,
        DRIVE_SOURCE_DATA_ARCHIVE,
        "dataset",
    )
else:
    DATA_DIR = str(LOCAL_MANUAL_DATA_ROOT)
    if not Path(DATA_DIR).is_dir():
        repository_manual_root = REPO_ROOT / "datasets/usa/detection_coco"
        if repository_manual_root.is_dir():
            shutil.copytree(repository_manual_root, LOCAL_MANUAL_DATA_ROOT, dirs_exist_ok=True)
            DATA_DIR = str(LOCAL_MANUAL_DATA_ROOT)
        else:
            print(
                "Manual scene COCO data is not ready yet. Upload source images and "
                "annotations to the Drive paths used in section 5, then enable conversion."
            )
    if Path(DATA_DIR).is_dir() and not DRIVE_SOURCE_DATA_ARCHIVE.is_file():
        archive_directory_to_drive(
            DATA_DIR,
            DRIVE_SOURCE_DATA_ARCHIVE,
            "dataset",
        )

if DATA_DIR:
    for split in ("train", "valid", "test"):
        try:
            print(split, _annotation_file(DATA_DIR, split))
        except FileNotFoundError as exc:
            print(exc)


## 5. Optional route: convert annotated full scenes to COCO

Use this route when you have complete vehicle or access-lane images with plate boxes annotated. Upload images to the private Drive folder `datasets/usa/manual_scene_source_images`. The notebook copies them to local `/content` for conversion. It places the annotation template at `datasets/usa/manual_scene_annotations.csv` in Drive; complete one row per plate box, leave negative-scene coordinates blank, and keep images from the same vehicle, camera sequence, or session in one `source_group`. The generated COCO set is archived to Drive and used locally. This route sets DATA_DIR to the generated COCO folder.

**Negative scenes and licensing.** The current 30 PKLot scenes are validation-only in this SIRAP split; they must not be moved into training. PKLot is licensed CC BY 4.0 and requires citation of its paper ([official license page](https://web.inf.ufpr.br/vri/databases/parking-lot-database/)). Its dataset card says license plates are not distinguishable and describes fixed elevated parking cameras in Brazil ([dataset card](https://docs.voxel51.com/dataset_zoo/datasets_hf/pklot.html)). These images are useful for a limited background false-detection check, but they do not represent the USA access camera. Before retraining, add separately reviewed, group-disjoint negative scenes from the target camera to the training set and validation set. Keep the reserved test split untouched.

In [ ]:
import shutil
from scripts.usa_annotations_to_coco import convert_annotations

MANUAL_DATA_ROOT = LOCAL_MANUAL_DATA_ROOT
SOURCE_IMAGES = MANUAL_DATA_ROOT / "source_images"
DRIVE_ANNOTATION_TEMPLATE = REPO_ROOT / "datasets/usa/detection_coco/annotations_template.csv"
ANNOTATIONS_CSV = DRIVE_MANUAL_ANNOTATIONS_CSV

SOURCE_IMAGES.mkdir(parents=True, exist_ok=True)
if not SOURCE_IMAGES.exists() or not any(SOURCE_IMAGES.iterdir()):
    if DRIVE_MANUAL_SOURCE_IMAGES.is_dir() and any(DRIVE_MANUAL_SOURCE_IMAGES.iterdir()):
        shutil.copytree(DRIVE_MANUAL_SOURCE_IMAGES, SOURCE_IMAGES, dirs_exist_ok=True)

if not ANNOTATIONS_CSV.exists() and DRIVE_ANNOTATION_TEMPLATE.is_file():
    shutil.copy2(DRIVE_ANNOTATION_TEMPLATE, ANNOTATIONS_CSV)
    print("Copied the annotation template to Drive:", ANNOTATIONS_CSV)
    print("Edit this CSV in Drive and add one row per plate box before conversion.")

RUN_COCO_CONVERSION = False
if RUN_COCO_CONVERSION:
    if not SOURCE_IMAGES.is_dir() or not any(SOURCE_IMAGES.iterdir()):
        raise FileNotFoundError(
            f"Upload full-scene images to the private Drive source folder first: {DRIVE_MANUAL_SOURCE_IMAGES}"
        )
    if not ANNOTATIONS_CSV.is_file():
        raise FileNotFoundError(f"Complete the annotation CSV in Drive first: {ANNOTATIONS_CSV}")
    # Work locally for speed; the generated COCO source is archived to Drive afterward.
    conversion_summary = convert_annotations(
        SOURCE_IMAGES, ANNOTATIONS_CSV, MANUAL_DATA_ROOT, verify_hashes=True
    )
    DATA_DIR = str(MANUAL_DATA_ROOT)
    SOURCE_DATASET_NAME = "Manually annotated USA scenes"
    ROBOFLOW_VERSION_USED = None
    archive_directory_to_drive(
        DATA_DIR,
        DRIVE_SOURCE_DATA_ARCHIVE,
        "dataset",
    )
    print(conversion_summary)
else:
    print(
        "Keep RUN_COCO_CONVERSION set to False until Drive contains full-scene images "
        "and a completed annotations CSV."
    )


## 6. Create the larger grouped holdout and prevent leakage

This cell consolidates the original preprocessing workflow. It reads COCO exports for train, validation, and test; resolves image paths; computes SHA-256 hashes; and groups augmentation variants by source filename. The optional source_group_manifest.csv can merge source files from the same vehicle or session. Exact duplicate files are merged across splits. Exact duplicates with conflicting boxes are excluded and listed in an audit CSV.

The original validation and test groups are held out. Additional source groups are reserved from training until the held-out set contains at least 400 images. A new validation subset is taken from the remaining training groups. The cell checks that no source group or exact image crosses train, validation, and test, then writes one-class COCO files, split manifests, and a JSON summary to local `/content`, archives the self-contained split to Drive, and publishes only its compact aggregate summary to GitHub. Without vehicle/session identifiers, the grouping guarantee is limited to source filename families and exact hashes.

On the next Colab run, section 4 restores the independent split archive from Drive to local `/content` and section 6 reuses it. Set `REBUILD_INDEPENDENT_SPLIT = True` to regenerate the split from the cached source dataset.

In [ ]:
import csv
import hashlib
import json
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path


MINIMUM_TEST_IMAGES = 400
SPLIT_SEED = 42

if DATASET_IS_CACHED_INDEPENDENT_SPLIT:
    INDEPENDENT_ROOT = Path(DATA_DIR).resolve()
    SOURCE_ROOT = INDEPENDENT_ROOT
    GROUP_MANIFEST = None
    summary_path = INDEPENDENT_ROOT / "split_summary.json"
    if not summary_path.is_file():
        raise FileNotFoundError(f"Cached split summary is missing: {summary_path}")
    SPLIT_SUMMARY = json.loads(summary_path.read_text(encoding="utf-8"))
    if SPLIT_SUMMARY.get("splits", {}).get("test", {}).get("images", 0) < MINIMUM_TEST_IMAGES:
        raise RuntimeError("The cached independent test split is smaller than the configured minimum.")
    SOURCE_DATASET_NAME = SPLIT_SUMMARY.get("source_dataset", SOURCE_DATASET_NAME)
    ROBOFLOW_VERSION_USED = SPLIT_SUMMARY.get("roboflow_version", ROBOFLOW_VERSION_USED)
    INDEPENDENT_TEST_READY = True
    print("Restored the audited independent split from Drive; split generation was skipped.")
    print(json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False))
    print("DATA_DIR:", DATA_DIR)
else:
    if not DATA_DIR:
        raise RuntimeError("Download the COCO dataset or set DATA_DIR first.")

    SOURCE_ROOT = Path(DATA_DIR).resolve()
    INDEPENDENT_ROOT = LOCAL_INDEPENDENT_ROOT
    GROUP_MANIFEST = SOURCE_ROOT / "source_group_manifest.csv"
    ADDITIONAL_TEST_GROUP_FRACTION = 0.15
    VALIDATION_GROUP_FRACTION = 0.15
    MINIMUM_TEST_IMAGES = 400
    SPLIT_SEED = 42

    if not SOURCE_ROOT.is_dir():
        raise FileNotFoundError(f"Dataset directory does not exist: {SOURCE_ROOT}")

    def source_key_from_filename(filename):
        basename = Path(filename).name
        return basename.split(".rf.", 1)[0] if ".rf." in basename else Path(basename).stem

    def resolve_source_image(source_folder, filename):
        basename = Path(filename).name
        candidates = [
            source_folder / filename,
            source_folder / basename,
            SOURCE_ROOT / filename,
            SOURCE_ROOT / basename,
        ]
        for candidate in candidates:
            if candidate.is_file():
                return candidate.resolve()
        matches = list(SOURCE_ROOT.rglob(basename))
        if matches:
            return matches[0].resolve()
        raise FileNotFoundError(f"Annotated image was not found: {filename}")

    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def annotation_signature(record):
        return sorted(
            tuple(round(float(value), 4) for value in annotation["bbox"])
            for annotation in record["annotations"]
        )

    source_payloads = {}
    source_records = {}
    category_maps = {}
    all_source_keys = set()

    for split_name in ("train", "valid", "test"):
        annotation_path = _annotation_file(SOURCE_ROOT, split_name)
        payload = json.loads(annotation_path.read_text(encoding="utf-8"))
        source_payloads[split_name] = payload
        category_maps[split_name] = {
            int(category["id"]): category.get("name", str(category["id"]))
            for category in payload.get("categories", [])
        }
        annotations_by_image = defaultdict(list)
        for annotation in payload.get("annotations", []):
            image_id = int(annotation["image_id"])
            if "bbox" not in annotation or len(annotation["bbox"]) != 4:
                raise ValueError(f"Annotation without a valid COCO bbox in {annotation_path}")
            annotations_by_image[image_id].append(annotation)

        image_ids = {int(image["id"]) for image in payload.get("images", [])}
        orphan_ids = set(annotations_by_image) - image_ids
        if orphan_ids:
            raise ValueError(f"Annotations have no matching image in {split_name}: {sorted(orphan_ids)[:5]}")

        records = []
        for image_record in payload.get("images", []):
            source_key = source_key_from_filename(image_record["file_name"])
            all_source_keys.add(source_key)
            record = {
                "source_split": split_name,
                "source_folder": annotation_path.parent,
                "image": image_record,
                "annotations": annotations_by_image.get(int(image_record["id"]), []),
                "source_key": source_key,
            }
            image_path = resolve_source_image(annotation_path.parent, image_record["file_name"])
            record["source_path"] = image_path
            record["sha256"] = sha256_file(image_path)
            records.append(record)
        source_records[split_name] = records

    # Create or extend the manifest so it remains useful after a new dataset export.
    if not GROUP_MANIFEST.is_file():
        with GROUP_MANIFEST.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=["source_key", "group_id"])
            writer.writeheader()
            writer.writerows({"source_key": key, "group_id": ""} for key in sorted(all_source_keys))
        print("Group manifest template created:", GROUP_MANIFEST)

    with GROUP_MANIFEST.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if not {"source_key", "group_id"}.issubset(reader.fieldnames or []):
            raise ValueError("The manifest must contain source_key and group_id columns.")
        manifest_rows = list(reader)

    manifest_by_key = {}
    for row in manifest_rows:
        key = (row.get("source_key") or "").strip()
        group_id = (row.get("group_id") or "").strip()
        if not key:
            continue
        if key in manifest_by_key and manifest_by_key[key] != group_id:
            raise ValueError(f"Duplicate source_key assigned to different groups: {key}")
        manifest_by_key[key] = group_id

    missing_keys = sorted(all_source_keys - set(manifest_by_key))
    if missing_keys:
        with GROUP_MANIFEST.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=["source_key", "group_id"])
            writer.writerows({"source_key": key, "group_id": ""} for key in missing_keys)
        print(f"Added {len(missing_keys)} new keys to the template.")

    for records in source_records.values():
        for record in records:
            manual_group = manifest_by_key.get(record["source_key"], "")
            record["base_group_id"] = (
                f"manual:{manual_group}" if manual_group else f"source:{record['source_key']}"
            )

    # The same exact file must never appear in different splits.
    parent = {
        record["base_group_id"]: record["base_group_id"]
        for records in source_records.values()
        for record in records
    }

    def find_group(group_id):
        while parent[group_id] != group_id:
            parent[group_id] = parent[parent[group_id]]
            group_id = parent[group_id]
        return group_id

    def union_groups(left, right):
        root_left, root_right = find_group(left), find_group(right)
        if root_left == root_right:
            return
        first, second = sorted((root_left, root_right))
        parent[second] = first

    records_by_hash = defaultdict(list)
    for records in source_records.values():
        for record in records:
            records_by_hash[record["sha256"]].append(record)

    conflicting_sha256 = set()
    ambiguous_duplicate_audit_rows = []
    for digest, copies in records_by_hash.items():
        signatures = {tuple(annotation_signature(record)) for record in copies}
        if len(signatures) > 1:
            conflicting_sha256.add(digest)
            for record in copies:
                ambiguous_duplicate_audit_rows.append({
                    "sha256": digest,
                    "source_split": record["source_split"],
                    "file_name": record["image"]["file_name"],
                    "source_key": record["source_key"],
                    "annotation_bboxes": json.dumps(
                        [annotation["bbox"] for annotation in record["annotations"]],
                        ensure_ascii=False,
                    ),
                })
        group_ids = sorted({record["base_group_id"] for record in copies})
        for group_id in group_ids[1:]:
            union_groups(group_ids[0], group_id)

    for records in source_records.values():
        for record in records:
            record["group_id"] = f"group:{find_group(record['base_group_id'])}"

    ambiguous_duplicate_records_removed = len(ambiguous_duplicate_audit_rows)
    for split_name, records in source_records.items():
        source_records[split_name] = [
            record for record in records if record["sha256"] not in conflicting_sha256
        ]
    print(
        "Exact duplicate images excluded due to conflicting bounding boxes:",
        ambiguous_duplicate_records_removed,
        "| hashes conflictivos:",
        len(conflicting_sha256),
    )

    rng = random.Random(SPLIT_SEED)

    def one_record_per_group(records):
        grouped = defaultdict(list)
        for record in records:
            grouped[record["group_id"]].append(record)
        selected = []
        for group_id in sorted(grouped):
            choices = sorted(grouped[group_id], key=lambda item: item["image"]["file_name"])
            selected.append(rng.choice(choices))
        return selected

    # The original validation and test splits are both held out for final testing.
    original_heldout = source_records["valid"] + source_records["test"]
    test_base_records = one_record_per_group(original_heldout)
    original_heldout_groups = {record["group_id"] for record in original_heldout}

    train_by_group = defaultdict(list)
    for record in source_records["train"]:
        if record["group_id"] not in original_heldout_groups:
            train_by_group[record["group_id"]].append(record)

    available_train_groups = sorted(train_by_group)
    if len(available_train_groups) < 3:
        raise ValueError("Fewer than three groups remain to create train, validation, and test splits.")

    rng.shuffle(available_train_groups)
    desired_extra_test_groups = max(
        round(len(available_train_groups) * ADDITIONAL_TEST_GROUP_FRACTION),
        max(0, MINIMUM_TEST_IMAGES - len(test_base_records)),
    )
    extra_test_group_count = min(desired_extra_test_groups, len(available_train_groups) - 2)
    if len(test_base_records) + extra_test_group_count < MINIMUM_TEST_IMAGES:
        raise ValueError(
            "There are not enough independent groups to create a test split of "
            f"{MINIMUM_TEST_IMAGES} images."
        )

    extra_test_group_ids = set(available_train_groups[:extra_test_group_count])
    remaining_train_group_ids = available_train_groups[extra_test_group_count:]
    extra_test_records = one_record_per_group([
        record
        for group_id in extra_test_group_ids
        for record in train_by_group[group_id]
    ])

    validation_group_count = max(
        1, round(len(remaining_train_group_ids) * VALIDATION_GROUP_FRACTION)
    )
    validation_group_count = min(validation_group_count, len(remaining_train_group_ids) - 1)
    validation_group_ids = set(remaining_train_group_ids[:validation_group_count])

    validation_records = one_record_per_group([
        record
        for group_id in validation_group_ids
        for record in train_by_group[group_id]
    ])
    training_records = [
        record
        for group_id in remaining_train_group_ids
        if group_id not in validation_group_ids
        for record in train_by_group[group_id]
    ]
    test_records = test_base_records + extra_test_records

    # Remove exact duplicates from train so they are not over-weighted.
    seen_train_hashes = set()
    deduplicated_training = []
    for record in sorted(training_records, key=lambda item: item["image"]["file_name"]):
        if record["sha256"] not in seen_train_hashes:
            seen_train_hashes.add(record["sha256"])
            deduplicated_training.append(record)
    duplicate_train_images_removed = len(training_records) - len(deduplicated_training)
    training_records = deduplicated_training

    split_records = {
        "train": training_records,
        "valid": validation_records,
        "test": test_records,
    }
    group_sets = {
        name: {record["group_id"] for record in records}
        for name, records in split_records.items()
    }
    hash_sets = {
        name: {record["sha256"] for record in records}
        for name, records in split_records.items()
    }
    for left, right in (("train", "valid"), ("train", "test"), ("valid", "test")):
        group_overlap = group_sets[left] & group_sets[right]
        hash_overlap = hash_sets[left] & hash_sets[right]
        if group_overlap or hash_overlap:
            raise RuntimeError(
                f"Leakage between {left} y {right}: "
                f"{len(group_overlap)} groups and {len(hash_overlap)} hashes."
            )

    if len(test_records) < MINIMUM_TEST_IMAGES:
        raise RuntimeError(f"The final test split contains only {len(test_records)} images.")

    marker = INDEPENDENT_ROOT / ".sirap_independent_split.json"
    if INDEPENDENT_ROOT.exists():
        if not marker.is_file():
            raise FileExistsError(
                f"{INDEPENDENT_ROOT} exists and was not created by this notebook; "
                "it will not be overwritten."
            )
        shutil.rmtree(INDEPENDENT_ROOT)
    INDEPENDENT_ROOT.mkdir(parents=True)

    source_licenses = next(
        (payload.get("licenses", []) for payload in source_payloads.values()
         if payload.get("licenses")),
        [],
    )
    manifest_rows = []
    split_summary = {}

    for split_name, records in split_records.items():
        split_folder = INDEPENDENT_ROOT / split_name
        split_folder.mkdir(parents=True, exist_ok=True)
        output_images = []
        output_annotations = []
        source_category_counts = Counter()
        annotation_id = 0

        for image_id, record in enumerate(
            sorted(records, key=lambda item: (item["source_split"], item["image"]["file_name"])),
            start=1,
        ):
            original_name = Path(record["image"]["file_name"]).name
            output_name = f"{record['source_split']}__{record['sha256'][:12]}__{original_name}"
            destination = split_folder / output_name
            try:
                destination.symlink_to(record["source_path"])
            except (OSError, NotImplementedError):
                shutil.copy2(record["source_path"], destination)

            image_out = dict(record["image"])
            image_out.update({
                "id": image_id,
                "file_name": output_name,
                "sirap_source_key": record["source_key"],
                "sirap_group_id": record["group_id"],
                "sirap_sha256": record["sha256"],
                "sirap_original_split": record["source_split"],
            })
            output_images.append(image_out)

            category_map = category_maps[record["source_split"]]
            for source_annotation in record["annotations"]:
                bbox = [float(value) for value in source_annotation["bbox"]]
                x, y, width, height = bbox
                if x < 0 or y < 0 or width <= 0 or height <= 0:
                    raise ValueError(f"Invalid COCO bbox in {original_name}: {bbox}")
                annotation_id += 1
                output_annotations.append({
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": 1,
                    "bbox": bbox,
                    "area": float(source_annotation.get("area", width * height)),
                    "iscrowd": int(source_annotation.get("iscrowd", 0)),
                })
                source_category_counts[
                    category_map.get(int(source_annotation["category_id"]), "unknown")
                ] += 1

            manifest_rows.append({
                "split": split_name,
                "file_name": output_name,
                "source_file_name": record["image"]["file_name"],
                "source_key": record["source_key"],
                "group_id": record["group_id"],
                "sha256": record["sha256"],
                "original_split": record["source_split"],
                "boxes": len(record["annotations"]),
            })

        empty_images = sum(
            1 for image in output_images
            if not any(annotation["image_id"] == image["id"] for annotation in output_annotations)
        )
        split_summary[split_name] = {
            "images": len(output_images),
            "groups": len({record["group_id"] for record in records}),
            "plates": len(output_annotations),
            "empty_images": empty_images,
            "source_category_annotations": dict(sorted(source_category_counts.items())),
        }
        coco_output = {
            "info": {
                "description": "SIRAP USA grouped internal test for RT-DETRv2-S",
                "source_dataset": SOURCE_DATASET_NAME,
                "grouping": "manual vehicle/session id when supplied; otherwise source filename and exact SHA-256",
            },
            "licenses": source_licenses,
            "images": output_images,
            "annotations": output_annotations,
            "categories": [{"id": 1, "name": "license_plate", "supercategory": "license_plate"}],
        }
        (split_folder / "_annotations.coco.json").write_text(
            json.dumps(coco_output, indent=2, ensure_ascii=False), encoding="utf-8"
        )

    with (INDEPENDENT_ROOT / "excluded_ambiguous_exact_duplicates.csv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=["sha256", "source_split", "file_name", "source_key", "annotation_bboxes"],
        )
        writer.writeheader()
        writer.writerows(ambiguous_duplicate_audit_rows)

    with (INDEPENDENT_ROOT / "split_manifest.csv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "split", "file_name", "source_file_name", "source_key",
                "group_id", "sha256", "original_split", "boxes",
            ],
        )
        writer.writeheader()
        writer.writerows(manifest_rows)

    train_group_overlap_with_original_holdout = {
        record["group_id"]
        for record in source_records["train"]
        if record["group_id"] in original_heldout_groups
    }
    SPLIT_SUMMARY = {
        "source_dataset": SOURCE_DATASET_NAME,
        "roboflow_version": ROBOFLOW_VERSION_USED,
        "seed": SPLIT_SEED,
        "minimum_test_images": MINIMUM_TEST_IMAGES,
        "additional_test_group_fraction": ADDITIONAL_TEST_GROUP_FRACTION,
        "validation_group_fraction": VALIDATION_GROUP_FRACTION,
        "test_design": (
            "one representative per group from original valid+test, plus one "
            "representative per randomly reserved train source group"
        ),
        "grouping_level": (
            "manual vehicle/session ids where supplied, otherwise source filename "
            "family; exact SHA-256 duplicates are merged"
        ),
        "manual_source_keys_grouped": sum(bool(value) for value in manifest_by_key.values()),
        "source_images": {name: len(records) for name, records in source_records.items()},
        "train_groups_removed_for_original_valid_test_overlap": len(train_group_overlap_with_original_holdout),
        "additional_train_groups_reserved_for_test": extra_test_group_count,
        "validation_groups_reserved_from_remaining_train": validation_group_count,
        "exact_duplicate_train_images_removed": duplicate_train_images_removed,
        "ambiguous_exact_duplicate_hashes_excluded": len(conflicting_sha256),
        "ambiguous_exact_duplicate_records_excluded": ambiguous_duplicate_records_removed,
        "ambiguous_duplicate_audit": str(INDEPENDENT_ROOT / "excluded_ambiguous_exact_duplicates.csv"),
        "unique_sha256_audited": len({record["sha256"] for records in split_records.values() for record in records}),
        "splits": split_summary,
    }
    (INDEPENDENT_ROOT / "split_summary.json").write_text(
        json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    marker.write_text(
        json.dumps({"generated_by": "SIRAP USA grouped COCO split", "seed": SPLIT_SEED}, indent=2),
        encoding="utf-8",
    )

    DATA_DIR = str(INDEPENDENT_ROOT)
    INDEPENDENT_TEST_READY = True
    print(json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False))
    print("New DATA_DIR for training and evaluation:", DATA_DIR)
    print("Optional vehicle/session manifest:", GROUP_MANIFEST)
    archive_directory_to_drive(
        INDEPENDENT_ROOT,
        DRIVE_INDEPENDENT_DATA_ARCHIVE,
        INDEPENDENT_ROOT.name,
    )
    if GROUP_MANIFEST and Path(GROUP_MANIFEST).is_file():
        persist_file_to_drive(
            GROUP_MANIFEST,
            DRIVE_DATA_ROOT / "manifests" / f"{DATASET_CACHE_KEY}_source_group_manifest.csv",
        )

SPLIT_SUMMARY_GITHUB_PATH = METRICS_DIR / "rtdetr_usa_independent_split_summary.json"
GITHUB_SPLIT_SUMMARY = dict(SPLIT_SUMMARY)
GITHUB_SPLIT_SUMMARY.pop("ambiguous_duplicate_audit", None)
SPLIT_SUMMARY_GITHUB_PATH.write_text(
    json.dumps(GITHUB_SPLIT_SUMMARY, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
publish_small_metrics(
    [SPLIT_SUMMARY_GITHUB_PATH],
    "Record USA independent split summary",
)


## 7. Validate the resulting COCO splits

This audit loads each split through SIRAP's COCO dataset class and checks annotation files, category count, image paths, and bounding-box dimensions. DATA_READY is true only when train and validation are usable and contain exactly one class. TEST_READY records whether the held-out split is available; this cell does not score the test data.

In [ ]:
from pathlib import Path
from src.train import CocoPlateDataset, _annotation_file

SPLITS = {}
DATA_READY = False
TEST_READY = False

if not DATA_DIR or not Path(DATA_DIR).is_dir():
    print("The COCO scene-data folder is not configured.")
    print("Set DATA_DIR and rerun this cell. OCR crop folders do not replace full-scene images with bounding boxes.")
else:
    for split in ("train", "val", "test"):
        try:
            dataset = CocoPlateDataset(DATA_DIR, split, image_processor=None)
            payload = json.loads(dataset.annotation_path.read_text(encoding="utf-8"))
            categories = payload.get("categories", [])
            missing_images, invalid_boxes = [], []
            for record in dataset.images:
                try:
                    image_path = dataset._image_path(record["file_name"])
                except FileNotFoundError:
                    missing_images.append(record["file_name"])
                    continue
                for ann in dataset.annotations_by_image.get(int(record["id"]), []):
                    x, y, w, h = map(float, ann["bbox"])
                    if w <= 0 or h <= 0 or x < 0 or y < 0:
                        invalid_boxes.append((record["file_name"], ann["bbox"]))
            SPLITS[split] = {
                "dataset": dataset,
                "payload": payload,
                "categories": categories,
                "images": len(dataset.images),
                "boxes": sum(len(v) for v in dataset.annotations_by_image.values()),
                "negative_images": sum(not dataset.annotations_by_image.get(int(item["id"]), []) for item in dataset.images),
                "missing_images": missing_images,
                "invalid_boxes": invalid_boxes,
            }
            print(
                f"{split}: {len(dataset.images)} images, "
                f"{SPLITS[split]['boxes']} boxes, "
                f"{len(categories)} categories, "
                f"{len(missing_images)} missing images, "
                f"{len(invalid_boxes)} invalid boxes"
            )
        except FileNotFoundError:
            print(f"No COCO annotation was found for {split}; train and valid are required, and test is required for final evaluation.")

    DATA_READY = (
        "train" in SPLITS and "val" in SPLITS
        and len(SPLITS["train"]["categories"]) == 1
        and len(SPLITS["val"]["categories"]) == 1
        and not SPLITS["train"]["missing_images"]
        and not SPLITS["val"]["missing_images"]
    )
    TEST_READY = "test" in SPLITS and not SPLITS["test"]["missing_images"]
    print("Data ready for training:", DATA_READY)
    print("Test split available:", TEST_READY)
    if not DATA_READY:
        print("Check that there is one plate category and that COCO files and image paths are valid before training.")

## 8. Visually inspect training annotations

The plot displays up to four training images with their COCO plate boxes in green. Inspect small plates, partial occlusions, glare, boxes that include too much background, and negative images with no boxes. If you find issues, correct the source annotations and rebuild the COCO splits before training.

In [ ]:
from src.data_processing import load_image

if DATA_READY:
    sample_data = SPLITS["train"]["dataset"]
    sample_count = min(4, len(sample_data.images))
    fig, axes = plt.subplots(1, sample_count, figsize=(5 * sample_count, 4))
    if sample_count == 1:
        axes = [axes]
    for ax, record in zip(axes, sample_data.images[:sample_count]):
        image_path = sample_data._image_path(record["file_name"])
        image = Image.fromarray(load_image(image_path.read_bytes()))
        draw = ImageDraw.Draw(image)
        for ann in sample_data.annotations_by_image.get(int(record["id"]), []):
            x, y, w, h = map(float, ann["bbox"])
            draw.rectangle((x, y, x + w, y + h), outline="lime", width=3)
        ax.imshow(image)
        ax.set_title(record["file_name"])
        ax.axis("off")
    plt.tight_layout()
else:
    print("Configure and validate DATA_DIR before reviewing examples.")

## 9. Recommended training protocol

The USA recipe uses a maximum of **50 epochs**, early-stopping patience **8**, a starting learning rate of 5e-5, weight decay 1e-4, and image size 640. Validation is evaluated once per epoch. The trainer retains the checkpoint with the lowest validation loss; the notebook then compares three independently trained seeds by validation mAP50-95 and copies the winner into the USA model directory.

Fifty epochs are a ceiling, not a requirement to complete every epoch. Patience 8 means training can stop after eight consecutive validation checks without improvement. Optuna is an optional screening stage with at most 12 epochs per trial; it uses validation mAP50-95 only. The detector test remains sealed until the checkpoint and threshold are frozen.

Use a GPU for fine-tuning. If memory is limited, start with batch size 4; do not alter the test set to accommodate training constraints.

## 10. Set training, search, and seed parameters

This is the single source of truth for the USA recipe. Three fixed seeds support a repeatability check; the notebook selects one model by validation mAP50-95 rather than averaging weights. MAX_TRIALS limits the Optuna budget. Record any changes in the final model-selection note.

The selected candidate is seed 1 (grouped Roboflow validation mAP50-95 = 0.8321). Its weights are frozen for this evaluation. The full-scene calibration did not meet the 0.99 precision target, so there is no operational threshold. Do not use the model for live or access decisions. Before retraining, add separately reviewed negative scenes to the training split; do not move the 30 validation-only PKLot images.

In [ ]:
EPOCHS_FINAL = 50
EARLY_STOPPING_PATIENCE = 8
MAX_TRIALS = 8
EPOCHS_TUNING = 12
FINAL_SEEDS = [0, 1, 2]
# The selected weights are frozen; an operational threshold is still unavailable.
MODEL_CHECKPOINT_FROZEN = True
MODEL_SELECTION_RECORD = "Seed 1 selected by grouped Roboflow validation (mAP50-95=0.8321266). Manual full-scene validation: 71 images (41 positive, 30 negative); best observed threshold=0.60 gave precision=0.9302326, recall=0.9756098 (TP=40, FP=3, FN=1), below target precision=0.99. Operational threshold is null and inference is blocked. The selected checkpoint's independent test has not been run."
MINIMUM_NEGATIVE_TRAIN_IMAGES = 1  # Presence check only; this is not a sufficient sample-size target.
DEFAULT_PARAMS = {
    "learning_rate": 5e-5,
    "weight_decay": 1e-4,
    "batch_size": BATCH_SIZE,
}
SELECTED_PARAMS = dict(DEFAULT_PARAMS)
print(SELECTED_PARAMS)

## 11. Optional Optuna search

This search compares learning rate, weight decay, and batch size using validation mAP50-95. It trains a short run for each trial, exports each candidate to OpenVINO IR, evaluates only on validation, and saves the best settings. RUN_OPTUNA is enabled in the configuration. Before running this cell, confirm the dataset passed the COCO audit and a GPU budget is available. It runs up to MAX_TRIALS trials using validation only; never tune against the test split.

In [ ]:
from src.train import train_detector
from src.evaluate import evaluate_detector
import optuna

RUN_OPTUNA = False
TUNING_DIR = RUNS_ROOT / "optuna"

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 5e-4, log=True)
    batch_choices = [4, 8] if GPU_MEMORY_GB >= 16 else [4]
    batch_size = trial.suggest_categorical("batch_size", batch_choices)
    output_dir = TUNING_DIR / f"trial_{trial.number}"
    train_detector(
        DATA_DIR, country=COUNTRY, model_id=MODEL_ID,
        output_dir=output_dir, run_dir=output_dir / "trainer",
        epochs=EPOCHS_TUNING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        batch_size=batch_size,
        learning_rate=learning_rate, weight_decay=weight_decay,
        seed=42, force=True,
    )
    metrics = evaluate_detector(
        DATA_DIR, country=COUNTRY, split="val",
        checkpoint=output_dir / "openvino", model_id=MODEL_ID,
        threshold=0.001,
        result_path=METRICS_DIR / f"optuna_usa_trial_{trial.number}.json",
    )
    score = metrics.get("map50_95")
    if score is None:
        raise RuntimeError("Validation did not produce mAP50-95.")
    return float(score)

if RUN_OPTUNA:
    if not DATA_READY:
        raise RuntimeError("Fix DATA_DIR before starting Optuna.")
    study = optuna.create_study(direction="maximize", study_name="rtdetrv2_s_usa")
    study.optimize(objective, n_trials=MAX_TRIALS)
    study.trials_dataframe().to_csv(METRICS_DIR / "rtdetr_usa_optuna_trials.csv", index=False)
    optuna_summary = {
        "country": COUNTRY,
        "model_id": MODEL_ID,
        "objective": "map50_95_validation",
        "best_value": study.best_value,
        "best_params": study.best_trial.params,
        "trial_count": len(study.trials),
        "maximum_epochs_per_trial": EPOCHS_TUNING,
        "detector": "RT-DETRv2-S",
        "inference_runtime": "OpenVINO Runtime",
    }
    optuna_summary_path = METRICS_DIR / "rtdetr_usa_optuna_summary.json"
    optuna_summary_path.write_text(
        json.dumps(optuna_summary, indent=2), encoding="utf-8"
    )
    publish_small_metrics(
        [
            *sorted(METRICS_DIR.glob("optuna_usa_trial_*.json")),
            METRICS_DIR / "rtdetr_usa_optuna_trials.csv",
            optuna_summary_path,
        ],
        "Publish USA Optuna metrics",
    )
    SELECTED_PARAMS = {
        "learning_rate": study.best_params["learning_rate"],
        "weight_decay": study.best_params["weight_decay"],
        "batch_size": study.best_params["batch_size"],
    }
    print("Best validation configuration:", SELECTED_PARAMS)
else:
    print("Optuna is disabled. The initial values will be used until the search is run.")

## 12. Train and select the final three-seed candidate

For each seed, train_detector fine-tunes the starting checkpoint with Transformers/PyTorch in local /content, evaluates once per epoch, applies early stopping, and exports both the Transformers checkpoint and OpenVINO IR. The notebook ranks the three candidates by validation mAP50-95, copies the winner under the mounted Drive model folder, and archives the selected run checkpoint on Drive. Compact candidate metrics are published to GitHub when GITHUB_TOKEN is set. RUN_FINAL_TRAINING is disabled by default. Before enabling it, add independently reviewed negative full-scene images to the training split and preserve separate validation and test groups. The training cell blocks a zero-negative training split; that presence check alone does not establish adequate diversity.

In [ ]:
RUN_FINAL_TRAINING = False

if RUN_FINAL_TRAINING:
    if not DATA_READY:
        raise RuntimeError("Training will not start: a valid COCO dataset is missing.")
    from src.train import CocoPlateDataset
    train_audit = CocoPlateDataset(DATA_DIR, "train", image_processor=None)
    val_audit = CocoPlateDataset(DATA_DIR, "val", image_processor=None)
    negative_train_images = sum(
        not train_audit.annotations_by_image.get(int(item["id"]), [])
        for item in train_audit.images
    )
    negative_val_images = sum(
        not val_audit.annotations_by_image.get(int(item["id"]), [])
        for item in val_audit.images
    )
    print("Negative full-scene images — train:", negative_train_images, "| validation:", negative_val_images)
    if negative_train_images < MINIMUM_NEGATIVE_TRAIN_IMAGES:
        raise RuntimeError(
            "Training is blocked: add independently reviewed negative full-scene images "
            "to train. Do not move the current validation negatives into train."
        )
    if negative_val_images < 1:
        raise RuntimeError(
            "Training is blocked: keep separately grouped negative full-scene images in validation."
        )

    # Preserve the existing blocked calibration until a new run passes the input checks.
    MODEL_CHECKPOINT_FROZEN = False
    MODEL_SELECTION_RECORD = ""
    old_threshold_path = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if old_threshold_path.is_file():
        from datetime import datetime, timezone
        archived_threshold_path = METRICS_DIR / (
            f"rtdetr_usa_threshold_calibration_previous_"
            f"{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.json"
        )
        old_threshold_path.rename(archived_threshold_path)
        print("Previous threshold calibration archived at:", archived_threshold_path)
    candidates = []
    for seed in FINAL_SEEDS:
        run_dir = RUNS_ROOT / f"seed_{seed}"
        train_detector(
            DATA_DIR, country=COUNTRY, model_id=MODEL_ID,
            output_dir=run_dir, run_dir=run_dir / "trainer",
            epochs=EPOCHS_FINAL,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            batch_size=SELECTED_PARAMS["batch_size"],
            learning_rate=SELECTED_PARAMS["learning_rate"],
            weight_decay=SELECTED_PARAMS["weight_decay"],
            seed=seed, force=True,
        )
        metrics = evaluate_detector(
            DATA_DIR, country=COUNTRY, split="val",
            checkpoint=run_dir / "openvino", model_id=MODEL_ID,
            threshold=0.001,
            result_path=METRICS_DIR / f"rtdetr_usa_val_seed_{seed}.json",
        )
        candidates.append({
            "seed": seed,
            "run_dir": str(run_dir),
            "map50_95": metrics.get("map50_95"),
            "map50": metrics.get("map50"),
            "recall": metrics.get("recall"),
            "precision": metrics.get("precision"),
        })

    ranked = [item for item in candidates if item["map50_95"] is not None]
    if not ranked:
        raise RuntimeError("No candidate produced mAP50-95.")
    best_run = max(ranked, key=lambda item: item["map50_95"])
    source_dir = Path(best_run["run_dir"])
    MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)

    if any(MODEL_OUTPUT.iterdir()):
        from datetime import datetime, timezone
        backup_archive = DRIVE_CHECKPOINTS_ROOT / (
            f"usa_previous_model_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.tar"
        )
        archive_directory_to_drive(MODEL_OUTPUT, backup_archive, "previous_model")
        shutil.rmtree(MODEL_OUTPUT)
        print("Previous selected model archived to Drive:", backup_archive)

    MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dir / "transformers", MODEL_OUTPUT / "transformers", dirs_exist_ok=True)
    shutil.copytree(source_dir / "openvino", MODEL_OUTPUT / "openvino", dirs_exist_ok=True)
    run_stamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
    selected_checkpoint_archive = DRIVE_CHECKPOINTS_ROOT / (
        f"rtdetr_usa_best_seed_{best_run['seed']}_{run_stamp}.tar"
    )
    archive_directory_to_drive(source_dir, selected_checkpoint_archive, f"seed_{best_run['seed']}")
    (METRICS_DIR / "rtdetr_usa_three_seed_validation.json").write_text(
        json.dumps({"candidates": candidates, "selected": best_run}, indent=2),
        encoding="utf-8",
    )
    validation_csv = METRICS_DIR / "rtdetr_usa_three_seed_validation.csv"
    pd.DataFrame(candidates).to_csv(validation_csv, index=False)
    selected_manifest = MODEL_OUTPUT / "openvino" / "sirap_training.json"
    english_manifest = METRICS_DIR / "rtdetr_usa_training_manifest.json"
    if selected_manifest.is_file():
        shutil.copy2(selected_manifest, english_manifest)
    print("USA model selected using validation:", best_run)
    print("Three-seed validation results:", validation_csv)
    print("Selected training manifest:", english_manifest if english_manifest.is_file() else "not found")
    print("Selected model and Transformers checkpoint saved to Drive:", MODEL_OUTPUT)
    print("Selected run checkpoint archive:", selected_checkpoint_archive)
    print("OpenVINO IR:", MODEL_OUTPUT / "openvino" / "openvino_model.xml")
    publish_small_metrics(
        [
            *sorted(METRICS_DIR.glob("rtdetr_usa_val_seed_*.json")),
            METRICS_DIR / "rtdetr_usa_three_seed_validation.json",
            METRICS_DIR / "rtdetr_usa_three_seed_validation.csv",
            english_manifest,
            METRICS_DIR / "entrenamiento_rtdetr.json",
        ],
        "Publish USA three-seed validation metrics",
    )
else:
    print("Final training is disabled. Set RUN_FINAL_TRAINING to True to start the three runs.")

## 13. Load the USA OpenVINO IR

The country-specific IR is preferred and loaded from the Drive model folder. If it is absent, this cell exports or loads the shared checkpoint from Drive for a smoke check; that shared model is only a technical fallback and does not establish USA performance. The detector is loaded on CPU through the native OpenVINO API. Fine-tuning must happen before this inference step.

In [ ]:
from src.export_openvino import export_openvino_model
from src.model import PlateDetector

COUNTRY_IR_DIR = MODEL_OUTPUT / "openvino"
SHARED_IR_DIR = DRIVE_SHARED_IR_DIR
COUNTRY_IR_XML = COUNTRY_IR_DIR / "openvino_model.xml"

if COUNTRY_IR_XML.is_file():
    DETECTOR_DIR = COUNTRY_IR_DIR
    MODEL_IS_COUNTRY_FINETUNED = True
else:
    DETECTOR_DIR = SHARED_IR_DIR
    MODEL_IS_COUNTRY_FINETUNED = False
    if not (SHARED_IR_DIR / "openvino_model.xml").is_file():
        export_openvino_model(MODEL_ID, SHARED_IR_DIR, image_size=IMAGE_SIZE)

detector = PlateDetector(
    checkpoint=DETECTOR_DIR,
    model_id=MODEL_ID,
    imgsz=IMAGE_SIZE,
    conf=0.001,
    class_id=0,
    device="CPU",
    fallback_checkpoint=None,
)
print("Loaded model:", detector.name)
print("Fine-tuned specifically for the USA:", MODEL_IS_COUNTRY_FINETUNED)
print("OpenVINO devices:", detector.core.available_devices)

## 14. Run a quick OpenVINO smoke check

A blank image is sent through the detector to confirm that the IR loads and inference returns without an exception. The number of boxes on a blank image is not an accuracy metric and must not be reported as model performance.

In [ ]:
blank_image = np.zeros((480, 640, 3), dtype=np.uint8)
smoke_detections = detector.detect(blank_image, max_detections=20)
print("OpenVINO inference completed.")
print("Boxes returned for the blank image:", len(smoke_detections))
print("This check only confirms that the model loads and runs; it does not measure accuracy.")

## 15. Calibrate the confidence threshold on validation

This cell reuses validation detections to compare confidence thresholds from 0.05 to 0.95 at IoU 0.50, including full-scene negatives. It selects the highest-recall threshold only when precision is at least 0.99 and true-positive support is at least 11. If no threshold qualifies, it records the best observed point for diagnosis, sets the operational threshold to null, and marks the model unusable for inference. It never falls back to 0.25 or to the best-F1 candidate.

In [ ]:
from src.data_processing import load_image


def box_iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-9)

def collect_detection_records(split, max_images=None, transform=None):
    dataset = CocoPlateDataset(DATA_DIR, split, image_processor=None)
    source_images = dataset.images if max_images is None else dataset.images[:max_images]
    records = []
    for item in source_images:
        image_path = dataset._image_path(item["file_name"])
        image = load_image(image_path.read_bytes())
        if transform is not None:
            image = transform(image)
        gt_boxes = []
        for ann in dataset.annotations_by_image.get(int(item["id"]), []):
            x, y, w, h = map(float, ann["bbox"])
            gt_boxes.append((x, y, x + w, y + h))
        detections = detector.detect(image, max_detections=300)
        records.append({
            "image_id": int(item["id"]),
            "predictions": [(d.box, d.confidence) for d in detections],
            "ground_truth": gt_boxes,
        })
    return records

def score_detection_records(records, threshold, iou_threshold=0.50):
    tp = fp = fn = 0
    for record in records:
        unmatched = set(range(len(record["ground_truth"])))
        predictions = sorted(
            (item for item in record["predictions"] if item[1] >= threshold),
            key=lambda item: item[1], reverse=True,
        )
        for box, _ in predictions:
            best_index, best_iou = None, 0.0
            for index in unmatched:
                value = box_iou_xyxy(box, record["ground_truth"][index])
                if value > best_iou:
                    best_index, best_iou = index, value
            if best_index is not None and best_iou >= iou_threshold:
                tp += 1
                unmatched.remove(best_index)
            else:
                fp += 1
        fn += len(unmatched)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

RUN_THRESHOLD_CALIBRATION = False
TARGET_PRECISION = 0.99
MINIMUM_TRUE_POSITIVES = 11
ACTIVE_THRESHOLD = None

if RUN_THRESHOLD_CALIBRATION:
    if not DATA_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Training data, validation data, and a USA fine-tuned IR are required for calibration.")
    if not MODEL_SELECTION_RECORD.strip():
        raise RuntimeError("Record the validation-selected checkpoint before calibrating its threshold.")

    val_records = collect_detection_records("val")
    threshold_rows = [
        {"threshold": round(float(value), 2), **score_detection_records(val_records, float(value))}
        for value in np.arange(0.05, 1.00, 0.05)
    ]
    threshold_table = pd.DataFrame(threshold_rows)
    eligible = threshold_table[
        (threshold_table["precision"] >= TARGET_PRECISION)
        & (threshold_table["tp"] >= MINIMUM_TRUE_POSITIVES)
    ]
    best_observed = threshold_table.sort_values(
        ["f1", "threshold"], ascending=[False, True]
    ).iloc[0]

    def calibration_row(row):
        return {
            "threshold": float(row["threshold"]),
            "tp": int(row["tp"]),
            "fp": int(row["fp"]),
            "fn": int(row["fn"]),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1"]),
        }

    if len(eligible):
        selected = eligible.sort_values(
            ["recall", "threshold"], ascending=[False, True]
        ).iloc[0]
        selected_threshold = float(selected["threshold"])
        usable_for_inference = True
        selection_rule = "highest recall subject to target precision and minimum true-positive support"
        selected_metrics = calibration_row(selected)
        ACTIVE_THRESHOLD = selected_threshold
    else:
        selected_metrics = calibration_row(best_observed)
        selected_threshold = None
        usable_for_inference = False
        selection_rule = "blocked: target precision or minimum true-positive support was not met"
        ACTIVE_THRESHOLD = None

    result = {
        "country": COUNTRY,
        "model": "RT-DETRv2-S",
        "runtime": "OpenVINO Runtime Python",
        "split": "manual_full_scene_valid",
        "calibration_images": len(val_records),
        "negative_validation_images": sum(not record["ground_truth"] for record in val_records),
        "validation_ground_truth_plates": sum(len(record["ground_truth"]) for record in val_records),
        "checkpoint": str(COUNTRY_IR_DIR),
        "model_selection_record": MODEL_SELECTION_RECORD,
        "iou_threshold": 0.50,
        "target_precision": TARGET_PRECISION,
        "minimum_true_positives": MINIMUM_TRUE_POSITIVES,
        "selection_rule": selection_rule,
        "threshold": selected_threshold,
        "usable_for_inference": usable_for_inference,
        "metrics": selected_metrics,
        "best_observed_candidate": calibration_row(best_observed),
    }
    threshold_metrics_path = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    threshold_metrics_path.write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )
    if PUBLISH_SMALL_METRICS_TO_GITHUB:
        publish_small_metrics(
            [threshold_metrics_path],
            "Publish USA validation threshold calibration",
        )
    display(threshold_table)
    print("Threshold selection:", selection_rule)
    if not usable_for_inference:
        print("No operational threshold was selected; downstream inference is blocked.")
    print("Validation threshold report:", result)
else:
    print("Calibration is disabled. Enable it only after the selected checkpoint and full-scene validation data are ready.")


## 16. Optional live mode: Cloudflare-published camera

This section adapts the former live-camera experiment to the current USA pipeline: RT-DETRv2-S detection through native OpenVINO Runtime, followed by the configured OCR reader. A camera in a private plant network is not directly reachable from a Colab runtime, so the bridge runs next to the camera and Cloudflare Tunnel carries HTTPS requests to it.

The mode is optional. It requires the bridge process, a tunnel, and the Colab Secrets described below. No camera connection is opened until the connection-check or live-session cell runs. Live processing is disabled by default. The four subsections create the bridge script, define a latest-frame reader and USA pipeline, check the connection, then run a bounded session that stages outputs locally and copies them to private Drive. This notebook does not track vehicles, decide access, open a barrier, or send live readings to SIRAP's private repository.


### 16.1 Generate the camera bridge for the camera PC

This cell writes a small Python bridge script into the temporary Colab workspace and copies that script to Drive for reuse; it does not connect to a camera or start a tunnel. Download the generated file to the PC that can reach the camera, install Python and OpenCV there, and start the bridge from a terminal.

The bridge supports a USB webcam, an IP camera that OpenCV can read, or a local video file. It binds to 127.0.0.1 by default and exposes three token-protected routes: /snapshot.jpg for one frame, /video for an MJPEG stream, and /status for camera health. It stores its random bearer token in a local file under the PC user's home directory so it survives restarts. The token is not written into this notebook.

1. Run the script on the camera PC, for example with source 0 for its first webcam. For an IP camera, supply the RTSP URL through a local environment variable rather than placing a camera password in notebook text or shell history.
2. Read the token from the local token file and add it to the Colab Secret named CAMARA_TOKEN.
3. On the same PC, start [Cloudflare Tunnel](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) with the local bridge address. A Quick Tunnel provides a temporary trycloudflare.com address without requiring an account or domain; use a named tunnel and Cloudflare Access for a stable, controlled deployment.
4. Add that base HTTPS address to the Colab Secret CAMARA_URL. Keep the address and token in Secrets; do not place either in notebook output or a public commit.

Install cloudflared by following [Cloudflare's official installation guide](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/downloads/), then run the command cloudflared tunnel --url http://127.0.0.1:8080. The URL changes when a Quick Tunnel is restarted. If the tunnel cannot connect, check that the camera PC can make outbound connections on port 7844 using QUIC/UDP or HTTP/2/TCP, as described in [Cloudflare's firewall guidance](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/configure-tunnels/tunnel-with-firewall/).

For a stable named tunnel, create a tunnel in the Cloudflare dashboard, add a public hostname such as camera.example.com that routes to http://127.0.0.1:8080, and protect it with a self-hosted Access application that requires a service token. Follow [Cloudflare's named-tunnel guide](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/local-management/create-local-tunnel/). The reader already sends the Access Client ID and Client Secret from Colab Secrets as request headers.

For a tunnel protected by Cloudflare Access, also add CF_ACCESS_CLIENT_ID and CF_ACCESS_CLIENT_SECRET to Colab Secrets. The bridge token remains required as a second check.

**Recommendation.** Use a Quick Tunnel only for a temporary experiment. Use a named tunnel with Cloudflare Access for a stable remote test. For routine production operation, place inference on an on-site computer and transmit only the required recognition events over an authenticated API; this avoids continuous public video streaming and reduces latency and bandwidth.

In [ ]:
from pathlib import Path

BRIDGE_SCRIPT_PATH = Path("/content/SIRAP_workspace/live_camera/bridge_camera.py")
BRIDGE_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)

BRIDGE_SCRIPT = r'''
import argparse
import cv2
import hmac
import json
import os
import secrets
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

class FrameHub:
    def __init__(self, source, max_side, quality, fps, loop):
        self.source = source
        self.max_side = max_side
        self.quality = quality
        self.delay = 1.0 / max(fps, 1)
        self.loop = loop
        self.lock = threading.Lock()
        self.jpeg = None
        self.frame_id = 0
        self.captured_at = None
        self.width = 0
        self.height = 0
        self.last_error = ""
        self.stop_event = threading.Event()

    def start(self):
        threading.Thread(target=self._capture_loop, daemon=True).start()

    def snapshot(self):
        with self.lock:
            return self.jpeg, self.captured_at, self.frame_id

    def status(self):
        with self.lock:
            return {
                "ready": self.jpeg is not None,
                "width": self.width,
                "height": self.height,
                "frame_id": self.frame_id,
                "captured_at": self.captured_at,
                "last_error": self.last_error,
            }

    def _capture_loop(self):
        while not self.stop_event.is_set():
            capture = cv2.VideoCapture(self.source)
            capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)
            if not capture.isOpened():
                with self.lock:
                    self.last_error = "OpenCV could not open the configured camera source."
                capture.release()
                self.stop_event.wait(2)
                continue

            source_is_file = isinstance(self.source, str) and Path(self.source).is_file()
            while not self.stop_event.is_set():
                started = time.monotonic()
                ok, frame = capture.read()
                if not ok:
                    if source_is_file and self.loop:
                        capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
                        continue
                    if not source_is_file:
                        with self.lock:
                            self.last_error = "Camera frame read failed; reconnecting."
                        break
                    with self.lock:
                        self.last_error = "The video file reached its end."
                    self.stop_event.set()
                    break

                height, width = frame.shape[:2]
                scale = min(1.0, self.max_side / max(height, width))
                if scale < 1.0:
                    frame = cv2.resize(
                        frame,
                        (max(1, int(width * scale)), max(1, int(height * scale))),
                        interpolation=cv2.INTER_AREA,
                    )
                ok, encoded = cv2.imencode(
                    ".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, self.quality]
                )
                if ok:
                    with self.lock:
                        self.jpeg = encoded.tobytes()
                        self.frame_id += 1
                        self.captured_at = time.time()
                        self.height, self.width = frame.shape[:2]
                        self.last_error = ""
                remaining = self.delay - (time.monotonic() - started)
                if remaining > 0:
                    self.stop_event.wait(remaining)
            capture.release()
            if not source_is_file and not self.stop_event.is_set():
                self.stop_event.wait(1)

class Handler(BaseHTTPRequestHandler):
    hub = None
    token = ""

    def log_message(self, format, *args):
        return

    def _authorized(self):
        supplied = self.headers.get("X-Camera-Token", "")
        return hmac.compare_digest(supplied, self.token)

    def _send_json(self, value, status=200):
        body = json.dumps(value).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if not self._authorized():
            self._send_json({"error": "unauthorized"}, 401)
            return

        if self.path == "/status":
            self._send_json(self.hub.status())
            return

        if self.path == "/snapshot.jpg":
            jpeg, captured_at, frame_id = self.hub.snapshot()
            if jpeg is None:
                self._send_json({"error": "camera frame is not ready"}, 503)
                return
            self.send_response(200)
            self.send_header("Content-Type", "image/jpeg")
            self.send_header("Content-Length", str(len(jpeg)))
            self.send_header("X-Frame-Id", str(frame_id))
            self.send_header("X-Captured-At", str(captured_at))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(jpeg)
            return

        if self.path == "/video":
            self.send_response(200)
            self.send_header("Content-Type", "multipart/x-mixed-replace; boundary=frame")
            self.send_header("Cache-Control", "no-store")
            self.send_header("Connection", "close")
            self.end_headers()
            try:
                last_id = -1
                while True:
                    jpeg, captured_at, frame_id = self.hub.snapshot()
                    if jpeg is None or frame_id == last_id:
                        time.sleep(0.02)
                        continue
                    last_id = frame_id
                    part = (
                        b"--frame\r\nContent-Type: image/jpeg\r\n"
                        + b"Content-Length: " + str(len(jpeg)).encode("ascii") + b"\r\n"
                        + b"X-Captured-At: " + str(captured_at).encode("ascii") + b"\r\n\r\n"
                        + jpeg + b"\r\n"
                    )
                    self.wfile.write(part)
                    self.wfile.flush()
            except (BrokenPipeError, ConnectionResetError, OSError):
                return
            return

        self._send_json({"error": "not found"}, 404)

def load_or_create_token(path):
    path = Path(path).expanduser()
    if path.exists():
        token = path.read_text(encoding="utf-8").strip()
        if token:
            return token, path
    path.parent.mkdir(parents=True, exist_ok=True)
    token = secrets.token_urlsafe(32)
    path.write_text(token + "\n", encoding="utf-8")
    try:
        path.chmod(0o600)
    except OSError:
        pass
    return token, path

def parse_source(value):
    return int(value) if value.isdigit() else value

def main():
    parser = argparse.ArgumentParser(description="Token-protected SIRAP camera bridge")
    parser.add_argument("--source", default=os.environ.get("SIRAP_CAMERA_SOURCE", "0"))
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=8080)
    parser.add_argument("--max-side", type=int, default=1920)
    parser.add_argument("--quality", type=int, default=85)
    parser.add_argument("--fps", type=float, default=15)
    parser.add_argument("--loop", action="store_true", help="Loop a local video file.")
    args = parser.parse_args()

    token, token_path = load_or_create_token(Path.home() / ".sirap_camera_token")
    hub = FrameHub(
        parse_source(args.source), max(320, args.max_side),
        min(100, max(30, args.quality)), max(1, args.fps), args.loop,
    )
    hub.start()
    Handler.hub = hub
    Handler.token = token
    server = ThreadingHTTPServer((args.host, args.port), Handler)
    server.daemon_threads = True
    print("Camera bridge listening on http://127.0.0.1:" + str(args.port))
    print("Token file:", token_path)
    print("Routes: /snapshot.jpg, /video, /status")
    print("Stop with Ctrl+C. Keep this terminal open while the tunnel is running.")
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        pass
    finally:
        hub.stop_event.set()
        server.server_close()

if __name__ == "__main__":
    main()
'''

BRIDGE_SCRIPT_PATH.write_text(BRIDGE_SCRIPT, encoding="utf-8")
persist_file_to_drive(
    BRIDGE_SCRIPT_PATH,
    DRIVE_SCRIPTS_DIR / "live_camera" / "bridge_camera.py",
)
print("Camera bridge script created:", BRIDGE_SCRIPT_PATH)
print("Download it from the Colab Files panel and copy it to the camera PC.")
print("On that PC, install OpenCV with: python -m pip install opencv-python")

### 16.2 Configure camera reading and the USA recognition pipeline

This cell defines the camera reader, frame-processing settings, and output folder. It does not connect to the camera or load the detector. The camera URL and token are read from Colab Secrets; optional Cloudflare Access service-token values are sent in request headers.

In automatic mode, an HTTP Cloudflare bridge is read as MJPEG first. If the stream does not deliver frames, the reader falls back to repeated snapshot requests. A background thread keeps only the newest frame so slower CPU inference does not build a growing queue. Use OpenCV mode for a source Colab can reach directly, such as a local video, RTSP stream, or HLS playlist.

The live reader uses the USA detector through native OpenVINO Runtime. Serial OCR keeps its ONNX model: ONNX Runtime selects OpenVINO Execution Provider on Intel CPUs and CPUExecutionProvider on other CPUs. It stores the OCR text as read plus an uppercase alphanumeric key for duplicate suppression; it does not force a country-specific plate format. A matching validation-threshold record is used when present. Otherwise, the provisional threshold is 0.25 and the cell reports that it was not calibrated. The notebook does not make authorization or barrier decisions.

Live detection requires a saved calibration with `usable_for_inference=true` and a numeric threshold. If validation misses the target, no provisional threshold is used and the live pipeline stops before opening the camera.

In [ ]:
from pathlib import Path
import os
import json
import threading
import time
from urllib.parse import urlsplit, urlunsplit

import cv2
import numpy as np
import requests
import yaml
from IPython.display import display
from PIL import Image
from src.data_processing import clean_text, crop_with_margin
from src.model import PlateDetector

def read_colab_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return str(value)
    except Exception:
        pass
    return os.environ.get(name, "")

LIVE_SOURCE = read_colab_secret("CAMARA_URL").strip()
LIVE_TOKEN = read_colab_secret("CAMARA_TOKEN").strip()
CF_ACCESS_CLIENT_ID = read_colab_secret("CF_ACCESS_CLIENT_ID").strip()
CF_ACCESS_CLIENT_SECRET = read_colab_secret("CF_ACCESS_CLIENT_SECRET").strip()

LIVE_READ_MODE = "auto"      # auto, mjpeg, snapshot, or opencv
LIVE_MJPEG_WAIT_S = 6
LIVE_SNAPSHOT_FPS = 4
LIVE_DURATION_S = 120        # None means run until the cell is interrupted.
LIVE_START_TIMEOUT_S = 30
LIVE_STALE_S = 20
LIVE_MAX_PROCESS_FPS = 5
LIVE_MAX_CANDIDATES = 5
LIVE_DISPLAY_FPS = 2
LIVE_CONFIRMATIONS = 2
LIVE_CONFIRM_WINDOW_S = 3
LIVE_DUPLICATE_COOLDOWN_S = 15
LIVE_SAVE_VIDEO = False
LIVE_OUTPUT_DIR = Path("/content/SIRAP_workspace/live_camera/usa")
LIVE_DRIVE_OUTPUT_DIR = DRIVE_PRIVATE_CAMERA_DIR
LIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class CameraUnavailable(RuntimeError):
    pass

def _camera_error(response):
    messages = {
        401: "The bridge rejected the camera token. Check the Colab Secret CAMARA_TOKEN.",
        403: "The request was denied by the bridge or Cloudflare Access. Check the configured Secrets.",
        404: "The bridge route was not found. Use the base tunnel URL in CAMARA_URL.",
        502: "Cloudflare could not reach the camera bridge. Confirm both processes are running.",
        530: "Cloudflare could not establish the requested connection. Check the tunnel and Access settings.",
    }
    return messages.get(
        response.status_code,
        "Camera request failed with HTTP " + str(response.status_code),
    )

class CameraReader:
    def __init__(self, source, token="", mode="auto", snapshot_fps=4,
                 access_client_id="", access_client_secret="", mjpeg_wait_s=6):
        self.source = source.strip()
        self.token = token
        self.mode = mode.lower()
        self.snapshot_fps = max(1.0, float(snapshot_fps))
        self.mjpeg_wait_s = max(1.0, float(mjpeg_wait_s))
        self.access_client_id = access_client_id
        self.access_client_secret = access_client_secret
        self.stop_event = threading.Event()
        self.lock = threading.Lock()
        self.frame = None
        self.received = 0
        self.last_frame_monotonic = None
        self.last_frame_epoch = None
        self.error = ""
        self.thread = None
        self.base_url = self._base_url(self.source)

    @staticmethod
    def _base_url(source):
        parts = urlsplit(source)
        if parts.scheme not in ("http", "https"):
            return source
        path = parts.path.rstrip("/")
        for suffix in ("/video", "/snapshot.jpg", "/status"):
            if path.endswith(suffix):
                path = path[:-len(suffix)]
                break
        return urlunsplit((parts.scheme, parts.netloc, path, "", "")).rstrip("/")

    def _headers(self):
        headers = {}
        if self.token:
            headers["X-Camera-Token"] = self.token
        if self.access_client_id:
            headers["CF-Access-Client-Id"] = self.access_client_id
        if self.access_client_secret:
            headers["CF-Access-Client-Secret"] = self.access_client_secret
        return headers

    def start(self):
        if not self.source:
            raise CameraUnavailable("No camera URL is configured. Add CAMARA_URL to Colab Secrets.")
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()
        return self

    def stop(self):
        self.stop_event.set()
        if self.thread:
            self.thread.join(timeout=3)

    def snapshot(self):
        with self.lock:
            if self.frame is None:
                return None, self.received, self.last_frame_monotonic, self.last_frame_epoch
            return self.frame.copy(), self.received, self.last_frame_monotonic, self.last_frame_epoch

    def _offer(self, frame):
        if frame is None or frame.size == 0:
            return
        with self.lock:
            self.frame = frame
            self.received += 1
            self.last_frame_monotonic = time.monotonic()
            self.last_frame_epoch = time.time()
            self.error = ""

    def _read_opencv(self):
        source = int(self.source) if self.source.isdigit() else self.source
        capture = cv2.VideoCapture(source)
        capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        if not capture.isOpened():
            capture.release()
            raise CameraUnavailable("OpenCV could not open the configured source.")
        try:
            while not self.stop_event.is_set():
                ok, frame = capture.read()
                if not ok:
                    raise CameraUnavailable("OpenCV stopped receiving frames from the source.")
                self._offer(frame)
        finally:
            capture.release()

    def _read_mjpeg(self):
        response = requests.get(
            self.base_url + "/video", headers=self._headers(),
            stream=True, timeout=(8, 4),
        )
        if response.status_code >= 400:
            message = _camera_error(response)
            response.close()
            raise CameraUnavailable(message)
        before = self.received
        started = time.monotonic()
        buffer = bytearray()
        jpeg_start = bytes([255, 216])
        jpeg_end = bytes([255, 217])
        try:
            for chunk in response.iter_content(chunk_size=2048):
                if self.stop_event.is_set():
                    break
                if not chunk:
                    continue
                buffer.extend(chunk)
                while True:
                    start = buffer.find(jpeg_start)
                    end = buffer.find(jpeg_end, start + 2) if start >= 0 else -1
                    if start < 0 or end < 0:
                        if len(buffer) > 5_000_000:
                            del buffer[:-2]
                        break
                    jpeg = bytes(buffer[start:end + 2])
                    del buffer[:end + 2]
                    frame = cv2.imdecode(np.frombuffer(jpeg, dtype=np.uint8), cv2.IMREAD_COLOR)
                    self._offer(frame)
                if self.received == before and time.monotonic() - started >= self.mjpeg_wait_s:
                    return False
        finally:
            response.close()
        return self.received > before

    def _read_snapshots(self):
        delay = 1.0 / self.snapshot_fps
        while not self.stop_event.is_set():
            started = time.monotonic()
            response = requests.get(
                self.base_url + "/snapshot.jpg",
                headers=self._headers(), timeout=(8, 10),
            )
            if response.status_code >= 400:
                message = _camera_error(response)
                response.close()
                raise CameraUnavailable(message)
            frame = cv2.imdecode(
                np.frombuffer(response.content, dtype=np.uint8), cv2.IMREAD_COLOR
            )
            response.close()
            if frame is None:
                raise CameraUnavailable("The snapshot endpoint did not return a valid JPEG.")
            self._offer(frame)
            self.stop_event.wait(max(0.0, delay - (time.monotonic() - started)))

    def _run(self):
        if self.mode == "opencv" or urlsplit(self.source).scheme not in ("http", "https"):
            try:
                self._read_opencv()
            except Exception as exc:
                self.error = str(exc)
            return

        while not self.stop_event.is_set():
            try:
                if self.mode == "snapshot":
                    self._read_snapshots()
                    return
                try:
                    received_mjpeg = self._read_mjpeg()
                except Exception as exc:
                    self.error = str(exc)
                    if self.mode != "auto":
                        return
                    received_mjpeg = False
                if self.mode == "mjpeg":
                    if not received_mjpeg:
                        self.error = "No MJPEG frames arrived before the timeout."
                        return
                    continue
                if not received_mjpeg:
                    self._read_snapshots()
                    return
            except Exception as exc:
                self.error = str(exc)
                if self.mode != "auto":
                    return
                self.stop_event.wait(2)

def open_camera():
    return CameraReader(
        LIVE_SOURCE, LIVE_TOKEN, LIVE_READ_MODE, LIVE_SNAPSHOT_FPS,
        CF_ACCESS_CLIENT_ID, CF_ACCESS_CLIENT_SECRET, LIVE_MJPEG_WAIT_S,
    )

def build_live_pipeline():
    if "DETECTOR_DIR" not in globals() or "MODEL_ID" not in globals():
        raise RuntimeError("Run section 13 first to select the USA OpenVINO IR.")
    if not Path(DETECTOR_DIR).exists():
        raise FileNotFoundError("The selected OpenVINO IR directory is missing.")

    threshold_path = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_path.is_file():
        raise RuntimeError("A validation threshold report is required before live inference.")
    record = json.loads(threshold_path.read_text(encoding="utf-8"))
    if not record.get("usable_for_inference", False) or record.get("threshold") is None:
        raise RuntimeError("Validation did not meet the operational target; live inference is blocked.")
    if (
        record.get("checkpoint") != str(COUNTRY_IR_DIR)
        or record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    live_confidence = float(record["threshold"])
    threshold_source = "validation calibration"

    usa_config = yaml.safe_load(
        (REPO_ROOT / "config/countries/usa.yaml").read_text(encoding="utf-8")
    )
    detector_config = usa_config["detector"]
    live_detector = PlateDetector(
        checkpoint=DETECTOR_DIR,
        model_id=MODEL_ID,
        imgsz=IMAGE_SIZE,
        conf=live_confidence,
        class_id=detector_config.get("class_id", 0),
        device="CPU",
        fallback_checkpoint=None,
    )
    # Use the global OCR reader; USA-specific plate validation rules are not applied here.
    reader = RuntimeSelectedSerialReader(model_name="cct-s-v2-global-model")
    print("Registration-serial OCR runtime:", reader.runtime_name)
    return live_detector, reader, live_confidence, threshold_source

def recognize_live_frame(live_detector, live_reader, image_rgb):
    detections = live_detector.detect(image_rgb, max_detections=LIVE_MAX_CANDIDATES)
    best = None
    for detection in detections[:LIVE_MAX_CANDIDATES]:
        crop = crop_with_margin(image_rgb, detection.box, 0.06)
        if crop is None or crop.size == 0:
            continue
        raw, confidence_min, confidence_mean = live_reader.read(crop)
        raw_text = str(raw or "").strip().upper()
        candidate = {
            "plate_text": raw_text,
            "plate_key": clean_text(raw_text),
            "ocr_confidence_min": float(confidence_min),
            "ocr_confidence_mean": float(confidence_mean),
            "det_confidence": float(detection.confidence),
            "box": detection.box,
        }
        if best is None or (
            candidate["ocr_confidence_min"], candidate["det_confidence"]
        ) > (
            best["ocr_confidence_min"], best["det_confidence"]
        ):
            best = candidate
        if candidate["plate_key"]:
            break
    return best

def annotate_live_frame(frame_bgr, recognition, camera_fps, process_fps):
    annotated = frame_bgr.copy()
    if recognition:
        box = recognition["box"]
        if box:
            x1, y1, x2, y2 = (int(round(value)) for value in box)
            height, width = annotated.shape[:2]
            x1, x2 = max(0, min(width - 1, x1)), max(0, min(width - 1, x2))
            y1, y2 = max(0, min(height - 1, y1)), max(0, min(height - 1, y2))
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 220, 0), 2)
            label = recognition["plate_text"] or "Plate detected; OCR unreadable"
            cv2.putText(
                annotated, str(label)[:32], (x1, max(24, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 0), 2, cv2.LINE_AA,
            )
    cv2.putText(
        annotated, f"Camera {camera_fps:.1f} fps | Processing {process_fps:.1f} fps",
        (12, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2, cv2.LINE_AA,
    )
    return annotated

print("Live-camera configuration defined; no connection has been opened.")
print("Camera URL configured:", bool(LIVE_SOURCE))
print("Reader mode:", LIVE_READ_MODE)
print("Local live-session staging:", LIVE_OUTPUT_DIR)
print("Private Drive destination:", LIVE_DRIVE_OUTPUT_DIR)
print("The selected model is USA fine-tuned:", globals().get("MODEL_IS_COUNTRY_FINETUNED", False))
if not globals().get("MODEL_IS_COUNTRY_FINETUNED", False):
    print("Warning: the shared checkpoint is a technical fallback, not evidence of USA accuracy.")

### 16.3 Test the camera connection

This check connects for five seconds without loading the detector or OCR model. It reports the number of frames received, approximate frame rate, resolution, and any connection error, then displays the last frame so the camera angle can be checked. It does not save a video or score recognition accuracy.

In [ ]:
LIVE_TEST_SECONDS = 5
RUN_CAMERA_CONNECTION_CHECK = False

if not RUN_CAMERA_CONNECTION_CHECK:
    print("Camera connection check is disabled. Set RUN_CAMERA_CONNECTION_CHECK = True to run the 5-second check.")
elif not LIVE_SOURCE:
    print("No camera URL is configured. Add CAMARA_URL and CAMARA_TOKEN to Colab Secrets, then rerun section 16.2.")
else:
    test_camera = open_camera()
    try:
        test_camera.start()
        start_time = time.monotonic()
        first_count = test_camera.received
        latest = None
        while time.monotonic() - start_time < LIVE_TEST_SECONDS:
            frame, count, _, _ = test_camera.snapshot()
            if frame is not None:
                latest = frame
            time.sleep(0.05)
        elapsed = max(time.monotonic() - start_time, 1e-6)
        final_count = test_camera.received
        if latest is None:
            detail = test_camera.error or "No frames arrived during the connection check."
            raise CameraUnavailable(detail)
        print({
            "frames_received": final_count - first_count,
            "approx_camera_fps": round((final_count - first_count) / elapsed, 2),
            "resolution": f"{latest.shape[1]} x {latest.shape[0]}",
            "reader_mode": LIVE_READ_MODE,
            "elapsed_seconds": round(elapsed, 2),
        })
        display(Image.fromarray(cv2.cvtColor(latest, cv2.COLOR_BGR2RGB)))
    except CameraUnavailable as exc:
        print("Camera connection check failed:", exc)
    finally:
        test_camera.stop()

### 16.4 Run the live session and save its outputs

Set LIVE_ENABLED to True only when you deliberately want to connect to the camera. The loop processes the newest available frame, applies the USA detector and OCR, and updates the preview. A reading is added to the session CSV after the same text is seen at least twice within the confirmation window; repeated readings are suppressed for the configured cooldown.

The session stops at LIVE_DURATION_S, when the camera has been stale for too long, or when you interrupt the cell. The output folder contains a CSV of confirmed text readings and a JSON session summary. Set LIVE_SAVE_VIDEO to True to also write an annotated MP4. The live loop does not use object tracking, an authorization list, or a gate action. It reports recognition evidence for review; it does not produce access decisions.

The CSV, summary, and optional video may contain plate data. The session writes locally while processing, then copies the completed files to the private Drive folder `private/usa/camera_sessions`. Keep that folder unshared and review the files only for an authorized purpose. The local staging copy remains in the temporary Colab workspace until the runtime ends.

In [ ]:
from collections import defaultdict, deque
from datetime import datetime, timezone
import csv
import json

LIVE_ENABLED = False

def run_live_session():
    live_detector, live_reader, live_confidence, threshold_source = build_live_pipeline()
    reader = open_camera()
    session_id = datetime.now(timezone.utc).strftime("usa_live_%Y%m%dT%H%M%S%fZ")
    session_dir = LIVE_OUTPUT_DIR / session_id
    drive_session_dir = LIVE_DRIVE_OUTPUT_DIR / session_id
    session_dir.mkdir(parents=True, exist_ok=False)
    readings_path = session_dir / "readings.csv"
    summary_path = session_dir / "summary.json"
    video_path = session_dir / "annotated.mp4"

    observations = defaultdict(lambda: deque(maxlen=LIVE_CONFIRMATIONS))
    last_logged = {}
    rows = []
    latencies_ms = []
    writer = None
    display_handle = None
    started = time.monotonic()
    last_processed_id = -1
    last_processed_at = 0.0
    last_display_at = 0.0
    processed = 0

    if LIVE_SAVE_VIDEO:
        print("Annotated video recording is enabled.")

    reader.start()
    try:
        initial_deadline = time.monotonic() + LIVE_START_TIMEOUT_S
        while reader.received == 0 and time.monotonic() < initial_deadline:
            if reader.error and LIVE_READ_MODE != "auto":
                raise CameraUnavailable(reader.error)
            time.sleep(0.1)
        if reader.received == 0:
            raise CameraUnavailable(reader.error or "No camera frame arrived before the startup timeout.")

        while True:
            now = time.monotonic()
            if LIVE_DURATION_S is not None and now - started >= LIVE_DURATION_S:
                break
            frame_bgr, frame_id, last_frame_at, captured_epoch = reader.snapshot()
            if frame_bgr is None:
                time.sleep(0.02)
                continue
            if last_frame_at is not None and now - last_frame_at > LIVE_STALE_S:
                raise CameraUnavailable("The camera stream stopped sending frames.")
            if frame_id == last_processed_id:
                time.sleep(0.01)
                continue
            if now - last_processed_at < 1.0 / max(LIVE_MAX_PROCESS_FPS, 0.1):
                time.sleep(0.01)
                continue

            last_processed_id = frame_id
            last_processed_at = now
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            inference_started = time.perf_counter()
            recognition = recognize_live_frame(live_detector, live_reader, frame_rgb)
            inference_ms = (time.perf_counter() - inference_started) * 1000
            latencies_ms.append(inference_ms)
            processed += 1

            elapsed_now = max(now - started, 1e-6)
            camera_fps = reader.received / elapsed_now
            process_fps = processed / elapsed_now
            annotated = annotate_live_frame(frame_bgr, recognition, camera_fps, process_fps)

            raw_text = recognition["plate_text"] if recognition else ""
            plate_key = recognition["plate_key"] if recognition else ""
            if plate_key:
                history = observations[plate_key]
                while history and now - history[0] > LIVE_CONFIRM_WINDOW_S:
                    history.popleft()
                history.append(now)
                last_time = last_logged.get(plate_key, 0.0)
                if len(history) >= LIVE_CONFIRMATIONS and now - last_time >= LIVE_DUPLICATE_COOLDOWN_S:
                    rows.append({
                        "timestamp_utc": datetime.fromtimestamp(
                            captured_epoch or time.time(), tz=timezone.utc
                        ).isoformat(),
                        "plate_text": raw_text,
                        "plate_key": plate_key,
                        "detector_confidence": recognition["det_confidence"],
                        "ocr_confidence_min": recognition["ocr_confidence_min"],
                        "ocr_confidence_mean": recognition["ocr_confidence_mean"],
                        "box_xyxy": json.dumps(recognition["box"]),
                        "confirmation_count": len(history),
                        "model": "RT-DETRv2-S",
                        "runtime": "OpenVINO Runtime",
                    })
                    last_logged[plate_key] = now

            if LIVE_SAVE_VIDEO:
                height, width = annotated.shape[:2]
                if writer is None:
                    writer = cv2.VideoWriter(
                        str(video_path), cv2.VideoWriter_fourcc(*"mp4v"),
                        max(1.0, min(float(LIVE_MAX_PROCESS_FPS), 10.0)), (width, height),
                    )
                    if not writer.isOpened():
                        writer.release()
                        writer = None
                        raise RuntimeError("Could not create the annotated MP4 output.")
                writer.write(annotated)

            if now - last_display_at >= 1.0 / max(LIVE_DISPLAY_FPS, 0.1):
                preview = Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
                if display_handle is None:
                    display_handle = display(preview, display_id=True)
                else:
                    display_handle.update(preview)
                last_display_at = now
    except KeyboardInterrupt:
        print("Live session interrupted by the user; saving the readings collected so far.")
    finally:
        reader.stop()
        if writer is not None:
            writer.release()

    elapsed = max(time.monotonic() - started, 1e-6)
    readings_df = pd.DataFrame(
        rows,
        columns=[
            "timestamp_utc", "plate_text", "plate_key", "detector_confidence",
            "ocr_confidence_min", "ocr_confidence_mean", "box_xyxy",
            "confirmation_count", "model", "runtime",
        ],
    )
    readings_df.to_csv(readings_path, index=False)
    summary = {
        "session_id": session_id,
        "country": "USA",
        "model": "RT-DETRv2-S",
        "runtime": "OpenVINO Runtime",
        "country_finetuned_checkpoint": bool(globals().get("MODEL_IS_COUNTRY_FINETUNED", False)),
        "confidence_threshold": live_confidence,
        "threshold_source": threshold_source,
        "duration_seconds": round(elapsed, 2),
        "frames_received": int(reader.received),
        "frames_processed": int(processed),
        "camera_fps": round(reader.received / elapsed, 3),
        "processing_fps": round(processed / elapsed, 3),
        "mean_pipeline_latency_ms": (
            round(float(np.mean(latencies_ms)), 2) if latencies_ms else None
        ),
        "confirmed_readings": len(rows),
        "readings_csv": str(drive_session_dir / "readings.csv"),
        "annotated_video": (
            str(drive_session_dir / "annotated.mp4")
            if LIVE_SAVE_VIDEO and video_path.is_file() else None
        ),
        "authorization_decisions": False,
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    persist_tree_to_drive(session_dir, drive_session_dir)
    print("Live session summary:")
    display(pd.Series(summary, name=session_id).to_frame())
    print("Confirmed readings:")
    safe_readings_preview = readings_df.drop(columns=["plate_text", "plate_key"], errors="ignore")
    display(safe_readings_preview)
    print("Local session staging:", session_dir)
    print("Private Drive session files:", drive_session_dir)
    if not globals().get("MODEL_IS_COUNTRY_FINETUNED", False):
        print("Reminder: this run used the shared technical fallback, not a USA-fine-tuned checkpoint.")
    return summary

live_summary = None
if not LIVE_ENABLED:
    print("Live processing is disabled. Set LIVE_ENABLED = True to run a deliberate camera session.")
elif not LIVE_SOURCE:
    print("Live processing skipped: add CAMARA_URL and CAMARA_TOKEN to Colab Secrets first.")
else:
    try:
        live_summary = run_live_session()
    except CameraUnavailable as exc:
        print("Live camera session stopped:", exc)


### 16.5 Generate the on-site edge inference runner

This is the second camera mode. The PC beside the camera runs RT-DETRv2-S through native OpenVINO Runtime. Serial OCR selects OpenVINO Execution Provider through ONNX Runtime on Intel CPUs and ONNX Runtime CPU on other CPUs; the OCR weights remain in ONNX format. Camera frames stay on that PC; the runner writes confirmed OCR text, confidence values, timestamps, and a camera identifier to a local CSV. It does not save or transmit video or plate crops.

The script requires the USA-fine-tuned OpenVINO IR. The notebook saves the selected Transformers checkpoint and OpenVINO IR under the mounted Drive model folder; copy the deployed IR into the project repository only when section 25 is explicitly used. It downloads the OCR ONNX artifact and YAML preprocessing config through fast-plate-ocr on first use. Pass the confidence threshold selected on the notebook's validation split. Install the backend matching the camera PC's CPU: `fast-plate-ocr[onnx-openvino]` on Intel or `fast-plate-ocr[onnx]` elsewhere, plus `openvino`, `numpy`, `opencv-python-headless`, `requests`, `PyYAML`, and SIRAP source dependencies. By default, edge-runner output is stored under the camera user's home folder, outside Colab and Google Drive. If that PC has Google Drive for desktop, set `SIRAP_EDGE_OUTPUT_DIR` to a private synchronized Drive folder; do not use a public folder for plate rows.

**Optional event upload is deliberately disabled.** The current SIRAP API does not provide a dedicated machine-to-machine USA event route. Its interactive operations endpoint is not a safe substitute: it represents an operator-confirmed action and may be connected to barrier logic. The generated runner accepts an optional HTTPS event endpoint only at the exact path /api/edge/events, with a bearer token; enable it only after that USA-aware endpoint is implemented and configured to check authorization on the server. The outgoing payload contains confirmed plate text and metadata only, never a frame or crop.

In [ ]:
from pathlib import Path

EDGE_RUNNER_PATH = Path("/content/SIRAP_workspace/live_camera/edge_inference_usa.py")
EDGE_RUNNER_PATH.parent.mkdir(parents=True, exist_ok=True)

EDGE_RUNNER_SCRIPT = r'''
import argparse
import csv
import json
import os
import sys
import threading
import time
import uuid
from collections import defaultdict, deque
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit

def locate_repository():
    hints = []
    if os.environ.get("SIRAP_REPO_ROOT"):
        hints.append(Path(os.environ["SIRAP_REPO_ROOT"]).expanduser())
    hints.append(Path.cwd())
    hints.extend(Path(__file__).resolve().parents)
    for candidate in hints:
        if (candidate / "src" / "model.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this script from inside a SIRAP_USA_ECUADOR clone or set SIRAP_REPO_ROOT."
    )

REPO_ROOT = locate_repository()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import cv2
import numpy as np
import requests
import yaml
from src.data_processing import clean_text, crop_with_margin
from src.model import PlateDetector

class RuntimeSelectedSerialReader:
    """ONNX OCR with OpenVINO EP on Intel CPU, standard ORT CPU on other CPUs."""

    def __init__(self, model_name="cct-s-v2-global-model"):
        import platform
        import re
        import onnxruntime as ort
        from fast_plate_ocr import LicensePlateRecognizer

        details = f"{platform.processor()} {platform.uname().processor}"
        cpuinfo = Path("/proc/cpuinfo")
        if cpuinfo.is_file():
            try:
                details += " " + cpuinfo.read_text(errors="ignore")
            except OSError:
                pass
        is_intel = bool(re.search(
            r"genuineintel|intel\(r\)|\bintel\b", details, re.I
        )) and not bool(re.search(r"authenticamd|\bamd\b", details, re.I))

        available = ort.get_available_providers()
        if is_intel:
            if "OpenVINOExecutionProvider" not in available:
                raise RuntimeError(
                    "Intel CPU detected, but OpenVINOExecutionProvider is missing. "
                    "Install fast-plate-ocr[onnx-openvino]."
                )
            providers = ["OpenVINOExecutionProvider"]
            if "CPUExecutionProvider" in available:
                providers.append("CPUExecutionProvider")
        else:
            if "CPUExecutionProvider" not in available:
                raise RuntimeError("Install fast-plate-ocr[onnx] for ONNX Runtime CPU inference.")
            providers = ["CPUExecutionProvider"]

        self.model = LicensePlateRecognizer(model_name, providers=providers)
        self.providers = self.model.model.get_providers()
        if is_intel and (
            not self.providers or self.providers[0] != "OpenVINOExecutionProvider"
        ):
            raise RuntimeError(
                f"OpenVINOExecutionProvider is not first in the Intel session: {self.providers}"
            )
        if not is_intel and (
            not self.providers or self.providers[0] != "CPUExecutionProvider"
        ):
            raise RuntimeError(f"ONNX Runtime CPU provider not active: {self.providers}")
        self.runtime_name = (
            "ONNX Runtime + OpenVINOExecutionProvider"
            if is_intel else "ONNX Runtime + CPUExecutionProvider"
        )

    def read(self, crop):
        prediction = self.model.run(
            np.ascontiguousarray(crop),
            return_confidence=True,
        )[0]
        text = clean_text(prediction.plate)
        probabilities = np.asarray(
            prediction.char_probs, dtype=np.float32
        ).ravel()[:max(len(text), 1)]
        if not probabilities.size:
            return text, 0.0, 0.0
        return text, float(probabilities.min()), float(probabilities.mean())

def parse_source(value):
    return int(value) if str(value).isdigit() else value

class LatestCamera:
    def __init__(self, source):
        self.source = parse_source(source)
        self.lock = threading.Lock()
        self.stop_event = threading.Event()
        self.frame = None
        self.frame_id = 0
        self.received_at = None
        self.error = ""

    def start(self):
        self.thread = threading.Thread(target=self._capture, daemon=True)
        self.thread.start()
        return self

    def stop(self):
        self.stop_event.set()
        self.thread.join(timeout=3)

    def latest(self):
        with self.lock:
            if self.frame is None:
                return None, self.frame_id, None
            return self.frame.copy(), self.frame_id, self.received_at

    def _capture(self):
        while not self.stop_event.is_set():
            camera = cv2.VideoCapture(self.source)
            camera.set(cv2.CAP_PROP_BUFFERSIZE, 1)
            if not camera.isOpened():
                self.error = "OpenCV could not open the configured camera."
                camera.release()
                self.stop_event.wait(2)
                continue
            while not self.stop_event.is_set():
                ok, frame = camera.read()
                if not ok:
                    self.error = "Camera frame read failed; reconnecting."
                    break
                with self.lock:
                    self.frame = frame
                    self.frame_id += 1
                    self.received_at = time.time()
                    self.error = ""
            camera.release()
            if not self.stop_event.is_set():
                self.stop_event.wait(1)

def recognize(detector, reader, image_rgb, max_candidates=5):
    detections = detector.detect(image_rgb, max_detections=max_candidates)
    best = None
    for detection in detections[:max_candidates]:
        crop = crop_with_margin(image_rgb, detection.box, 0.06)
        if crop is None or crop.size == 0:
            continue
        raw, confidence_min, confidence_mean = reader.read(crop)
        raw_text = str(raw or "").strip().upper()
        candidate = {
            "plate_text": raw_text,
            "plate_key": clean_text(raw_text),
            "detector_confidence": float(detection.confidence),
            "ocr_confidence_min": float(confidence_min),
            "ocr_confidence_mean": float(confidence_mean),
            "box": tuple(float(value) for value in detection.box),
        }
        if best is None or (
            candidate["ocr_confidence_min"], candidate["detector_confidence"]
        ) > (
            best["ocr_confidence_min"], best["detector_confidence"]
        ):
            best = candidate
        if candidate["plate_key"]:
            break
    return best

def send_event_if_enabled(event, enabled):
    if not enabled:
        return "local_only"
    endpoint = os.environ.get("SIRAP_EDGE_EVENTS_URL", "").strip()
    token = os.environ.get("SIRAP_EDGE_EVENTS_TOKEN", "").strip()
    if not endpoint or not token:
        raise RuntimeError(
            "Event upload needs SIRAP_EDGE_EVENTS_URL and SIRAP_EDGE_EVENTS_TOKEN."
        )
    parsed = urlsplit(endpoint)
    if parsed.scheme != "https":
        raise ValueError("The edge event endpoint must use HTTPS.")
    if parsed.path.rstrip("/") != "/api/edge/events":
        raise ValueError(
            "For safety, uploads are restricted to the dedicated /api/edge/events route."
        )
    response = requests.post(
        endpoint,
        json=event,
        headers={"Authorization": "Bearer " + token},
        timeout=(5, 10),
    )
    response.raise_for_status()
    return "sent"

def main():
    usa_config = yaml.safe_load(
        (REPO_ROOT / "config" / "countries" / "usa.yaml").read_text(encoding="utf-8")
    )
    detector_config = usa_config["detector"]
    checkpoint = (REPO_ROOT / detector_config["checkpoint"]).resolve()
    parser = argparse.ArgumentParser(
        description="Run USA plate detection locally beside the camera."
    )
    parser.add_argument(
        "--source", default=os.environ.get("SIRAP_CAMERA_SOURCE", "0"),
        help="Local camera index, local video path, or source URL.",
    )
    parser.add_argument(
        "--threshold", type=float,
        default=float(os.environ.get("SIRAP_USA_CONFIDENCE", "0.25")),
        help="Use the confidence threshold selected on USA validation.",
    )
    parser.add_argument("--max-fps", type=float, default=5)
    parser.add_argument("--duration", type=float, default=0,
                        help="Seconds to run; 0 means run until interrupted.")
    parser.add_argument("--min-confirmations", type=int, default=2)
    parser.add_argument("--confirm-window", type=float, default=3)
    parser.add_argument("--duplicate-cooldown", type=float, default=15)
    parser.add_argument("--camera-id", default=os.environ.get("SIRAP_CAMERA_ID", "usa-edge-01"))
    parser.add_argument("--preview", action="store_true")
    parser.add_argument(
        "--upload-events", action="store_true",
        help="Send confirmed text events to a dedicated HTTPS USA intake endpoint.",
    )
    args = parser.parse_args()

    if not 0 <= args.threshold <= 1:
        raise ValueError("--threshold must be between 0 and 1.")
    if not checkpoint.is_dir():
        raise FileNotFoundError(
            "The USA fine-tuned OpenVINO folder is missing: " + str(checkpoint)
        )
    if not any(checkpoint.glob("*.xml")) or not any(checkpoint.glob("*.bin")):
        raise FileNotFoundError(
            "Copy both OpenVINO IR files (.xml and .bin) into " + str(checkpoint)
        )

    detector = PlateDetector(
        checkpoint=checkpoint,
        model_id=detector_config["model_id"],
        imgsz=int(detector_config.get("inference_size", 640)),
        conf=args.threshold,
        class_id=int(detector_config.get("class_id", 0)),
        device="CPU",
        fallback_checkpoint=None,
    )
    reader = RuntimeSelectedSerialReader()
    print("Registration-serial OCR backend:", reader.runtime_name)
    camera = LatestCamera(args.source).start()
    output_dir = Path(os.environ.get("SIRAP_EDGE_OUTPUT_DIR", str(Path.home() / ".sirap" / "live" / "usa"))).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)
    session_id = datetime.now(timezone.utc).strftime("usa_edge_%Y%m%dT%H%M%SZ")
    output_path = output_dir / (session_id + "_readings.csv")
    fieldnames = [
        "event_id", "timestamp_utc", "country", "camera_id", "plate_text",
        "plate_key", "detector_confidence", "ocr_confidence_min",
        "ocr_confidence_mean", "model", "runtime", "upload_status",
    ]
    history = defaultdict(lambda: deque(maxlen=max(1, args.min_confirmations)))
    last_sent = {}
    last_frame_id = -1
    last_process_at = 0.0
    started = time.monotonic()
    processed = 0
    rows_written = 0
    last_frame_at = None
    source = str(args.source)
    print("USA model:", detector.model_path)
    print("Threshold:", args.threshold)
    print("Camera source:", source if source.isdigit() else "(configured source)")
    print("Event upload:", "enabled" if args.upload_events else "disabled; local CSV only")
    if args.threshold == 0.25:
        print("Set the calibrated USA validation threshold before production use.")

    try:
        with output_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            while True:
                now = time.monotonic()
                if args.duration and now - started >= args.duration:
                    break
                frame_bgr, frame_id, received_at = camera.latest()
                if frame_bgr is None:
                    if camera.error:
                        print(camera.error)
                    if now - started > 30:
                        raise RuntimeError("No camera frame arrived within 30 seconds.")
                    time.sleep(0.05)
                    continue
                if frame_id == last_frame_id:
                    if last_frame_at and now - last_frame_at > 20:
                        raise RuntimeError("The camera stream has stopped.")
                    time.sleep(0.01)
                    continue
                if now - last_process_at < 1.0 / max(args.max_fps, 0.1):
                    time.sleep(0.01)
                    continue

                last_frame_id = frame_id
                last_process_at = now
                last_frame_at = now
                image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                result = recognize(detector, reader, image_rgb)
                processed += 1

                if result and result["plate_key"]:
                    key = result["plate_key"]
                    samples = history[key]
                    while samples and now - samples[0] > args.confirm_window:
                        samples.popleft()
                    samples.append(now)
                    if (
                        len(samples) >= max(1, args.min_confirmations)
                        and now - last_sent.get(key, 0) >= args.duplicate_cooldown
                    ):
                        event = {
                            "event_id": str(uuid.uuid4()),
                            "timestamp_utc": datetime.fromtimestamp(
                                received_at or time.time(), tz=timezone.utc
                            ).isoformat(),
                            "country": "USA",
                            "camera_id": args.camera_id,
                            "plate_text": result["plate_text"],
                            "plate_key": key,
                            "detector_confidence": result["detector_confidence"],
                            "ocr_confidence_min": result["ocr_confidence_min"],
                            "ocr_confidence_mean": result["ocr_confidence_mean"],
                            "model": "RT-DETRv2-S",
                            "runtime": "OpenVINO Runtime",
                        }
                        try:
                            upload_status = send_event_if_enabled(event, args.upload_events)
                        except Exception as exc:
                            upload_status = "failed:" + type(exc).__name__
                            print("Event upload failed; the local row is still saved:", exc)
                        writer.writerow({**event, "upload_status": upload_status})
                        handle.flush()
                        rows_written += 1
                        last_sent[key] = now
                        print("Confirmed plate text:", result["plate_text"], "|", upload_status)

                if args.preview:
                    shown = frame_bgr.copy()
                    if result and result.get("box"):
                        x1, y1, x2, y2 = (int(round(value)) for value in result["box"])
                        cv2.rectangle(shown, (x1, y1), (x2, y2), (0, 220, 0), 2)
                        cv2.putText(shown, result["plate_text"] or "Plate detected", (x1, max(24, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 0), 2, cv2.LINE_AA)
                    cv2.imshow("SIRAP USA edge inference", shown)
                    if cv2.waitKey(1) & 0xFF == ord("q"):
                        break
    except KeyboardInterrupt:
        print("Edge session interrupted; the local CSV has been saved.")
    finally:
        camera.stop()
        if args.preview:
            cv2.destroyAllWindows()

    elapsed = max(time.monotonic() - started, 1e-6)
    print(json.dumps({
        "session_id": session_id,
        "frames_received": camera.frame_id,
        "frames_processed": processed,
        "confirmed_readings": rows_written,
        "elapsed_seconds": round(elapsed, 2),
        "processing_fps": round(processed / elapsed, 3),
        "local_csv": str(output_path),
        "video_uploaded": False,
    }, indent=2))

if __name__ == "__main__":
    main()
'''

EDGE_RUNNER_PATH.write_text(EDGE_RUNNER_SCRIPT, encoding="utf-8")
persist_file_to_drive(
    EDGE_RUNNER_PATH,
    DRIVE_SCRIPTS_DIR / "live_camera" / "edge_inference_usa.py",
)
print("On-site edge runner created:", EDGE_RUNNER_PATH)
print("A reusable copy was saved to Drive:", DRIVE_SCRIPTS_DIR / "live_camera" / "edge_inference_usa.py")
print("Download it from Colab Files and place it in the local project clone before running it.")

## 17. Run reproducible image robustness checks

This diagnostic applies synthetic low-light, high-light, blur, half-resolution, and JPEG-compression changes to up to 200 validation images. It compares precision, recall, F1, and counts at the validation-selected threshold. The detector test split is not used here, so it remains available for the final one-time evaluation. Synthetic stress conditions do not replace real USA scenes across weather, lighting, distance, camera angles, and plate designs.

In [ ]:
import cv2

def darker(image):
    return np.clip(image.astype(np.float32) * 0.55, 0, 255).astype(np.uint8)

def brighter(image):
    return np.clip(image.astype(np.float32) * 1.25 + 12, 0, 255).astype(np.uint8)

def blurred(image):
    return cv2.GaussianBlur(image, (5, 5), 0)

def reduced(image):
    h, w = image.shape[:2]
    small = cv2.resize(image, (max(1, w // 2), max(1, h // 2)))
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

def compressed(image):
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    ok, encoded = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 35])
    if not ok:
        return image
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    return cv2.cvtColor(decoded, cv2.COLOR_BGR2RGB)

RUN_ROBUSTNESS_TEST = False
ACTIVE_THRESHOLD = None
if RUN_ROBUSTNESS_TEST:
    if not DATA_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("A COCO validation split and the USA fine-tuned IR are required.")
    threshold_file = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_file.is_file():
        raise RuntimeError("Calibrate the USA confidence threshold on validation first.")
    threshold_record = json.loads(threshold_file.read_text(encoding="utf-8"))
    if (
        threshold_record.get("checkpoint") != str(COUNTRY_IR_DIR)
        or threshold_record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    if not threshold_record.get("usable_for_inference", False) or threshold_record.get("threshold") is None:
        raise RuntimeError("Validation did not meet the operational target; threshold-based inference and evaluation are blocked.")
    ACTIVE_THRESHOLD = float(threshold_record["threshold"])
    variants = {
        "original": None,
        "low_light": darker,
        "high_light": brighter,
        "blur": blurred,
        "half_resolution": reduced,
        "jpeg_compression": compressed,
    }
    stress_rows = []
    for name, transform in variants.items():
        records = collect_detection_records("val", max_images=200, transform=transform)
        stress_rows.append({"condition": name, **score_detection_records(records, ACTIVE_THRESHOLD)})
    robustness_table = pd.DataFrame(stress_rows)
    display(robustness_table)
    robustness_metrics_path = METRICS_DIR / "image_robustness_rtdetr_usa.csv"
    robustness_table.to_csv(robustness_metrics_path, index=False)
    publish_small_metrics(
        [robustness_metrics_path],
        "Publish USA image robustness summary",
    )
else:
    print("Stress testing is disabled. It does not change the model or production threshold.")

## 18. Measure CPU inference latency

This timing check measures detector-only inference on up to 200 validation images after warming up with the first ten. It reports median, p95, p99, and approximate frames per second from the median. Image reading, OCR, network transfer, and application logic are excluded. Use the same CPU class and input size when comparing these figures with the deployed Space.

In [ ]:
RUN_LATENCY_TEST = False
if RUN_LATENCY_TEST:
    if not DATA_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("A COCO validation split and the USA fine-tuned IR are required.")
    validation_dataset = SPLITS["val"]["dataset"]
    samples = validation_dataset.images[:min(200, len(validation_dataset.images))]
    for item in samples[:10]:
        image = load_image(validation_dataset._image_path(item["file_name"]).read_bytes())
        detector.detect(image, max_detections=100)
    latencies_ms = []
    for item in samples[10:]:
        image = load_image(validation_dataset._image_path(item["file_name"]).read_bytes())
        start = time.perf_counter()
        detector.detect(image, max_detections=100)
        latencies_ms.append((time.perf_counter() - start) * 1000)
    latency_summary = {
        "n": len(latencies_ms),
        "device": "CPU",
        "median_ms": float(np.median(latencies_ms)) if latencies_ms else None,
        "p95_ms": float(np.percentile(latencies_ms, 95)) if latencies_ms else None,
        "p99_ms": float(np.percentile(latencies_ms, 99)) if latencies_ms else None,
        "fps_from_median": 1000 / float(np.median(latencies_ms)) if latencies_ms else None,
    }
    latency_metrics_path = METRICS_DIR / "openvino_usa_latency.json"
    latency_metrics_path.write_text(
        json.dumps(latency_summary, indent=2), encoding="utf-8"
    )
    publish_small_metrics(
        [latency_metrics_path],
        "Publish USA OpenVINO latency summary",
    )
    print(latency_summary)
else:
    print("Latency measurement is disabled.")

## 19. Read and evaluate the two USA plate fields

A useful U.S. plate identifier keeps two OCR results separate: the issuing jurisdiction and the registration serial. The serial by itself does not identify a jurisdiction-qualified registration record. This section reads the printed state name, maps it to a two-letter abbreviation, and reads the registration serial with the plate OCR model. When both are available, it forms a key such as STATE_CODE:SERIAL.

### Plate-size context and crop quality

Most standard U.S. passenger-vehicle plates use a 12 × 6 inch format, a 2:1 width-to-height reference. Motorcycle and specialized plates may use other dimensions. Perspective, rotation, crop margins, and plate type change the pixel ratio in a camera image, so the notebook records the ratio as a diagnostic and does not reject a plate because it differs from 2:1. OCR upscaling preserves the crop's aspect ratio rather than stretching characters.

### Separate jurisdiction-name and serial OCR

Tesseract scans the plate crop and its upper and lower text bands for the printed state name. It compares recognized text against the English names of all 50 states and maps a match to the corresponding two-letter abbreviation. D.C. is also included because it occurs in the labeled data. If the scan finds only a clear two-letter jurisdiction code, it maps that code to the long name. Ambiguous state text is left blank rather than guessed. Serial OCR runs separately through ONNX Runtime: Intel CPUs prioritize OpenVINOExecutionProvider, while non-Intel CPUs use CPUExecutionProvider.

The input crop dataset is archived to Drive and restored to local /content for processing. The detailed per-crop CSV contains filenames, state-name and abbreviation predictions, registration serials, confidence values, and lookup keys. It is written to the private Drive OCR folder. The notebook preview omits serials and filenames. Only aggregate coverage and accuracy metrics are written to GitHub results/metrics/.

The dataset ground truth labels jurisdictions by abbreviation, not by the exact printed state-name text. Therefore, the report measures jurisdiction-code accuracy; it does not claim exact state-name transcription accuracy.

In [ ]:
RUN_OCR_CROP_TEST = False
RUN_SEPARATE_FIELD_OCR = False  # Set True to OCR the state name and serial as separate fields.
OCR_SOURCE_REPO_DIR = REPO_ROOT / "datasets/usa/seg_and_ocr"
OCR_DRIVE_ARCHIVE = DRIVE_DATA_ARCHIVE_DIR / "usa_seg_and_ocr_source.tar"
OCR_LOCAL_CACHE_PARENT = LOCAL_DATA_CACHE_ROOT / "seg_and_ocr_cache"
if OCR_DRIVE_ARCHIVE.is_file():
    OCR_LOCAL_ROOT = restore_directory_archive(
        OCR_DRIVE_ARCHIVE,
        OCR_LOCAL_CACHE_PARENT,
        "seg_and_ocr",
    )
elif OCR_SOURCE_REPO_DIR.is_dir():
    archive_directory_to_drive(
        OCR_SOURCE_REPO_DIR,
        OCR_DRIVE_ARCHIVE,
        "seg_and_ocr",
    )
    OCR_LOCAL_ROOT = restore_directory_archive(
        OCR_DRIVE_ARCHIVE,
        OCR_LOCAL_CACHE_PARENT,
        "seg_and_ocr",
    )
else:
    raise FileNotFoundError(f"USA OCR source dataset is missing: {OCR_SOURCE_REPO_DIR}")

OCR_CSV = OCR_LOCAL_ROOT / "groundtruth.csv"
OCR_IMAGE_DIR = OCR_LOCAL_ROOT / "usimages"
FIELD_OCR_OUTPUT_PATH = DRIVE_PRIVATE_OCR_DIR / "rtdetr_usa_plate_fields_ocr.csv"
FIELD_OCR_LOCAL_OUTPUT_PATH = LOCAL_PRIVATE_OCR_DIR / "rtdetr_usa_plate_fields_ocr.csv"
OCR_CROP_OUTPUT_PATH = DRIVE_PRIVATE_OCR_DIR / "rtdetr_usa_crop_ocr.csv"
OCR_CROP_LOCAL_OUTPUT_PATH = LOCAL_PRIVATE_OCR_DIR / "rtdetr_usa_crop_ocr.csv"
FIELD_OCR_SUMMARY_PATH = METRICS_DIR / "rtdetr_usa_plate_fields_ocr_summary.json"

if RUN_OCR_CROP_TEST or RUN_SEPARATE_FIELD_OCR:
    import re
    import shutil
    import subprocess
    from difflib import SequenceMatcher

    import cv2
    from tqdm.auto import tqdm
    from src.data_processing import clean_text, plate_distance

    if not OCR_CSV.is_file() or not OCR_IMAGE_DIR.is_dir():
        raise FileNotFoundError("Could not find the USA profile CSV or OCR crop images.")

    # This map contains every U.S. state abbreviation and English long name.
    # D.C. is added because it is present in the labeled crop dataset.
    STATE_ABBR_TO_NAME = {
        "AL": "Alabama",
        "AK": "Alaska",
        "AZ": "Arizona",
        "AR": "Arkansas",
        "CA": "California",
        "CO": "Colorado",
        "CT": "Connecticut",
        "DE": "Delaware",
        "FL": "Florida",
        "GA": "Georgia",
        "HI": "Hawaii",
        "ID": "Idaho",
        "IL": "Illinois",
        "IN": "Indiana",
        "IA": "Iowa",
        "KS": "Kansas",
        "KY": "Kentucky",
        "LA": "Louisiana",
        "ME": "Maine",
        "MD": "Maryland",
        "MA": "Massachusetts",
        "MI": "Michigan",
        "MN": "Minnesota",
        "MS": "Mississippi",
        "MO": "Missouri",
        "MT": "Montana",
        "NE": "Nebraska",
        "NV": "Nevada",
        "NH": "New Hampshire",
        "NJ": "New Jersey",
        "NM": "New Mexico",
        "NY": "New York",
        "NC": "North Carolina",
        "ND": "North Dakota",
        "OH": "Ohio",
        "OK": "Oklahoma",
        "OR": "Oregon",
        "PA": "Pennsylvania",
        "RI": "Rhode Island",
        "SC": "South Carolina",
        "SD": "South Dakota",
        "TN": "Tennessee",
        "TX": "Texas",
        "UT": "Utah",
        "VT": "Vermont",
        "VA": "Virginia",
        "WA": "Washington",
        "WV": "West Virginia",
        "WI": "Wisconsin",
        "WY": "Wyoming",
        "DC": "District of Columbia",
    }
    STATE_NAME_TO_ABBR = {
        re.sub(r"[^A-Z]", "", name.upper()): code
        for code, name in STATE_ABBR_TO_NAME.items()
    }

    def normalize_ocr_word(value):
        return re.sub(r"[^A-Z]", "", str(value).upper())

    def match_state_name(tokens, confidences):
        candidates = []
        for code, state_name in STATE_ABBR_TO_NAME.items():
            name_tokens = [normalize_ocr_word(part) for part in state_name.split()]
            target = " ".join(name_tokens)
            min_width = max(1, len(name_tokens) - 1)
            max_width = min(3, len(name_tokens) + 1)
            for width in range(min_width, max_width + 1):
                for start in range(max(0, len(tokens) - width + 1)):
                    phrase_tokens = tokens[start:start + width]
                    phrase = " ".join(phrase_tokens)
                    score = SequenceMatcher(None, phrase, target).ratio()
                    if score == 1.0 or score >= 0.84:
                        token_confidence = float(
                            np.mean(confidences[start:start + width])
                        ) if width else 0.0
                        candidates.append({
                            "code": code,
                            "name": state_name,
                            "score": score,
                            "confidence": token_confidence,
                            "width": width,
                        })

        if not candidates:
            return None

        exact = [candidate for candidate in candidates if candidate["score"] == 1.0]
        if exact:
            return max(exact, key=lambda candidate: (candidate["width"], candidate["confidence"]))

        best_by_code = {}
        for candidate in candidates:
            code = candidate["code"]
            if code not in best_by_code or (
                candidate["score"], candidate["confidence"]
            ) > (
                best_by_code[code]["score"], best_by_code[code]["confidence"]
            ):
                best_by_code[code] = candidate

        ranked = sorted(
            best_by_code.values(),
            key=lambda candidate: (candidate["score"], candidate["confidence"]),
            reverse=True,
        )
        if not ranked or ranked[0]["score"] < 0.84 or ranked[0]["confidence"] < 30:
            return None
        if len(ranked) > 1 and ranked[0]["score"] - ranked[1]["score"] < 0.08:
            return None
        return ranked[0]

    def read_usa_jurisdiction(crop_rgb):
        if not shutil.which("tesseract"):
            if shutil.which("apt-get"):
                subprocess.run(["apt-get", "update", "-qq"], check=True)
                subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr"], check=True)
            else:
                raise RuntimeError(
                    "Tesseract is not installed. Install the Tesseract OCR executable, then rerun this cell."
                )

        import pytesseract

        image = np.asarray(crop_rgb)
        if image.ndim != 3 or image.shape[2] < 3 or image.size == 0:
            return {
                "jurisdiction_name": "",
                "jurisdiction_abbreviation": "",
                "jurisdiction_raw_ocr": "",
                "jurisdiction_ocr_source": "",
                "jurisdiction_match_score": 0.0,
                "tesseract_confidence_mean": 0.0,
            }

        height, width = image.shape[:2]
        gray = cv2.cvtColor(image[:, :, :3], cv2.COLOR_RGB2GRAY)
        gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
        top_end = max(1, int(round(height * 0.45)))
        bottom_start = min(height - 1, int(round(height * 0.55)))
        views = [
            ("full_plate", gray, 11),
            ("upper_band", gray[:top_end, :], 7),
            ("lower_band", gray[bottom_start:, :], 7),
        ]

        view_results = []
        name_matches = []
        for view_name, view, page_mode in views:
            if view.size == 0:
                continue
            scale = max(2, min(5, int(round(180 / max(1, view.shape[0])))))
            enlarged = cv2.resize(
                view,
                None,
                fx=scale,
                fy=scale,
                interpolation=cv2.INTER_CUBIC,
            )
            try:
                data = pytesseract.image_to_data(
                    enlarged,
                    config=(
                        f"--oem 1 --psm {page_mode} "
                        "-c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
                    ),
                    output_type=pytesseract.Output.DICT,
                )
            except pytesseract.TesseractError:
                # A single malformed or unusually small crop must not stop the full dataset scan.
                continue
            words = []
            confidences = []
            for raw_word, raw_confidence in zip(data["text"], data["conf"]):
                word = normalize_ocr_word(raw_word)
                try:
                    confidence = float(raw_confidence)
                except (TypeError, ValueError):
                    confidence = -1.0
                if word and confidence >= 0:
                    words.append(word)
                    confidences.append(confidence)

            view_text = " ".join(words)
            view_results.append({
                "view": view_name,
                "text": view_text,
                "words": words,
                "confidences": confidences,
            })
            candidate = match_state_name(words, confidences)
            if candidate is not None:
                candidate["source"] = view_name
                name_matches.append(candidate)

        raw_text = " | ".join(
            f"{result['view']}: {result['text']}"
            for result in view_results
            if result["text"]
        )

        exact_matches = [
            candidate for candidate in name_matches if candidate["score"] == 1.0
        ]
        if exact_matches:
            selected = max(
                exact_matches,
                key=lambda candidate: (candidate["width"], candidate["confidence"]),
            )
            return {
                "jurisdiction_name": selected["name"],
                "jurisdiction_abbreviation": selected["code"],
                "jurisdiction_raw_ocr": raw_text,
                "jurisdiction_ocr_source": selected["source"],
                "jurisdiction_match_score": float(selected["score"]),
                "tesseract_confidence_mean": float(selected["confidence"]),
            }

        ranked_matches = sorted(
            name_matches,
            key=lambda candidate: (candidate["score"], candidate["confidence"]),
            reverse=True,
        )
        if ranked_matches:
            selected = ranked_matches[0]
            second_score = (
                ranked_matches[1]["score"]
                if len(ranked_matches) > 1
                and ranked_matches[1]["code"] != selected["code"]
                else 0.0
            )
            if (
                selected["score"] >= 0.84
                and selected["confidence"] >= 30
                and selected["score"] - second_score >= 0.08
            ):
                return {
                    "jurisdiction_name": selected["name"],
                    "jurisdiction_abbreviation": selected["code"],
                    "jurisdiction_raw_ocr": raw_text,
                    "jurisdiction_ocr_source": selected["source"],
                    "jurisdiction_match_score": float(selected["score"]),
                    "tesseract_confidence_mean": float(selected["confidence"]),
                }

        # Some plate designs show a two-letter jurisdiction code instead of a full name.
        # Only inspect the upper/lower bands and require a single high-confidence code.
        band_codes = []
        for result in view_results:
            if result["view"] not in ("upper_band", "lower_band"):
                continue
            for word, confidence in zip(result["words"], result["confidences"]):
                if word in STATE_ABBR_TO_NAME and len(word) == 2 and confidence >= 70:
                    band_codes.append((word, confidence, result["view"]))
        unique_codes = sorted({code for code, _, _ in band_codes})
        if len(unique_codes) == 1:
            code = unique_codes[0]
            confidence, source = max(
                [(confidence, source) for candidate_code, confidence, source in band_codes
                 if candidate_code == code],
                key=lambda item: item[0],
            )
            return {
                "jurisdiction_name": STATE_ABBR_TO_NAME[code],
                "jurisdiction_abbreviation": code,
                "jurisdiction_raw_ocr": raw_text,
                "jurisdiction_ocr_source": source,
                "jurisdiction_match_score": 1.0,
                "tesseract_confidence_mean": float(confidence),
            }

        return {
            "jurisdiction_name": "",
            "jurisdiction_abbreviation": "",
            "jurisdiction_raw_ocr": raw_text,
            "jurisdiction_ocr_source": "",
            "jurisdiction_match_score": 0.0,
            "tesseract_confidence_mean": 0.0,
        }

    ocr_truth = pd.read_csv(
        OCR_CSV, header=None, names=["image_file", "state", "plate_gt"], dtype=str
    ).dropna(subset=["image_file", "plate_gt"])
    recognizer = RuntimeSelectedSerialReader()
    print("Registration-serial OCR backend:", recognizer.runtime_name)
    ocr_rows = []

    if RUN_SEPARATE_FIELD_OCR:
        display(pd.DataFrame([
            {"state_abbreviation": code, "state_long_name": name}
            for code, name in sorted(STATE_ABBR_TO_NAME.items())
        ]))

    for row in tqdm(ocr_truth.itertuples(index=False), total=len(ocr_truth), desc="USA separate-field OCR"):
        image_path = OCR_IMAGE_DIR / str(row.image_file)
        if not image_path.is_file():
            continue

        crop = np.asarray(Image.open(image_path).convert("RGB"))
        serial_prediction = recognizer.run(crop, return_confidence=True)[0]
        serial_pred = clean_text(serial_prediction.plate)
        serial_gt = clean_text(row.plate_gt)
        serial_probs = np.asarray(serial_prediction.char_probs, dtype=np.float32).ravel()

        state_gt_raw = str(row.state).strip().upper()
        state_gt_code = (
            state_gt_raw
            if state_gt_raw in STATE_ABBR_TO_NAME
            else STATE_NAME_TO_ABBR.get(normalize_ocr_word(state_gt_raw), "")
        )
        jurisdiction = (
            read_usa_jurisdiction(crop)
            if RUN_SEPARATE_FIELD_OCR
            else {
                "jurisdiction_name": "",
                "jurisdiction_abbreviation": "",
                "jurisdiction_raw_ocr": "",
                "jurisdiction_ocr_source": "",
                "jurisdiction_match_score": 0.0,
                "tesseract_confidence_mean": 0.0,
            }
        )

        crop_height, crop_width = crop.shape[:2]
        aspect_ratio = float(crop_width / max(crop_height, 1))
        state_code_pred = jurisdiction["jurisdiction_abbreviation"]
        state_match = bool(state_gt_code and state_code_pred == state_gt_code)
        serial_match = serial_pred == serial_gt
        lookup_key = (
            f"{state_code_pred}:{serial_pred}"
            if state_code_pred and serial_pred
            else ""
        )

        ocr_rows.append({
            "image_file": str(row.image_file),
            "state_gt_code": state_gt_code,
            "state_gt_long_name": STATE_ABBR_TO_NAME.get(state_gt_code, ""),
            "state_name_pred": jurisdiction["jurisdiction_name"],
            "state_abbreviation_pred": state_code_pred,
            "jurisdiction_name_pred": jurisdiction["jurisdiction_name"],
            "jurisdiction_abbreviation_pred": state_code_pred,
            "jurisdiction_raw_ocr": jurisdiction["jurisdiction_raw_ocr"],
            "jurisdiction_ocr_source": jurisdiction["jurisdiction_ocr_source"],
            "jurisdiction_match_score": jurisdiction["jurisdiction_match_score"],
            "tesseract_confidence_mean": jurisdiction["tesseract_confidence_mean"],
            "registration_serial_gt": serial_gt,
            "registration_serial_pred": serial_pred,
            "serial_exact_match": serial_match,
            "serial_char_errors": plate_distance(serial_pred, serial_gt),
            "serial_gt_characters": len(serial_gt),
            "serial_confidence_min": (
                float(serial_probs.min()) if serial_probs.size else 0.0
            ),
            "jurisdiction_code_match": state_match,
            "combined_identifier_match": bool(state_match and serial_match),
            "lookup_key": lookup_key,
            "plate_crop_aspect_ratio": aspect_ratio,
            "distance_from_2_to_1_reference": abs(aspect_ratio - 2.0),
            # Legacy serial-only names retained for the existing crop OCR report.
            "state": state_gt_code,
            "plate_gt": serial_gt,
            "plate_pred": serial_pred,
            "exact_match": serial_match,
            "char_errors": plate_distance(serial_pred, serial_gt),
            "gt_characters": len(serial_gt),
        })

    ocr_results = pd.DataFrame(ocr_rows)
    if ocr_results.empty:
        raise RuntimeError("No labeled plate crops were found; no OCR results were produced.")

    if RUN_OCR_CROP_TEST:
        exact_match_rate = float(ocr_results["serial_exact_match"].mean())
        character_error_rate = float(
            ocr_results["serial_char_errors"].sum()
            / max(ocr_results["serial_gt_characters"].sum(), 1)
        )
        crop_summary = {
            "n": len(ocr_results),
            "registration_serial_exact_match_rate": exact_match_rate,
            "registration_serial_character_error_rate": character_error_rate,
        }
        print(crop_summary)
        display(
            ocr_results.groupby("state_gt_code")["serial_exact_match"]
            .agg(["count", "mean"])
            .sort_values("mean")
        )
        ocr_results.to_csv(OCR_CROP_LOCAL_OUTPUT_PATH, index=False)
        persist_file_to_drive(OCR_CROP_LOCAL_OUTPUT_PATH, OCR_CROP_OUTPUT_PATH)
        crop_summary_path = METRICS_DIR / "rtdetr_usa_crop_ocr_summary.json"
        crop_summary_path.write_text(json.dumps(crop_summary, indent=2), encoding="utf-8")
        publish_small_metrics(
            [crop_summary_path],
            "Publish USA crop OCR aggregate metrics",
        )

    if RUN_SEPARATE_FIELD_OCR:
        valid_jurisdiction = ocr_results["state_gt_code"].isin(STATE_ABBR_TO_NAME)
        field_summary = {
            "n": len(ocr_results),
            "jurisdiction_code_coverage": float(
                ocr_results["jurisdiction_abbreviation_pred"].ne("").mean()
            ),
            "jurisdiction_code_accuracy_all_labeled_crops": float(
                ocr_results.loc[valid_jurisdiction, "jurisdiction_code_match"].mean()
            ) if valid_jurisdiction.any() else 0.0,
            "jurisdiction_code_accuracy_when_read": float(
                ocr_results.loc[
                    valid_jurisdiction
                    & ocr_results["jurisdiction_abbreviation_pred"].ne(""),
                    "jurisdiction_code_match",
                ].mean()
            ) if (
                valid_jurisdiction
                & ocr_results["jurisdiction_abbreviation_pred"].ne("")
            ).any() else 0.0,
            "registration_serial_exact_match_rate": float(
                ocr_results["serial_exact_match"].mean()
            ),
            "combined_jurisdiction_and_serial_exact_match_rate": float(
                ocr_results["combined_identifier_match"].mean()
            ),
            "state_long_name_word_accuracy": (
                "Not measured: ground truth contains state abbreviations, not the printed state-name text."
            ),
        }
        print(field_summary)
        # Keep plate serials and source filenames out of notebook/GitHub output.
        safe_preview = ocr_results[[
            "state_name_pred",
            "state_abbreviation_pred",
            "jurisdiction_abbreviation_pred",
            "plate_crop_aspect_ratio",
        ]].head(25)
        display(safe_preview)
        ocr_results.to_csv(FIELD_OCR_LOCAL_OUTPUT_PATH, index=False)
        persist_file_to_drive(FIELD_OCR_LOCAL_OUTPUT_PATH, FIELD_OCR_OUTPUT_PATH)
        FIELD_OCR_SUMMARY_PATH.write_text(
            json.dumps(field_summary, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        publish_small_metrics(
            [FIELD_OCR_SUMMARY_PATH],
            "Publish USA field OCR aggregate metrics",
        )
        print("Private field-level OCR table saved to Drive:", FIELD_OCR_OUTPUT_PATH)
else:
    print("USA crop OCR is disabled. Set RUN_OCR_CROP_TEST or RUN_SEPARATE_FIELD_OCR to True.")



# Optional diagnostic using the notebook's runtime-selected ONNX serial reader.
OCR_SPOTCHECK_CSV_PATH = None  # CSV with image_path and plate_gt columns.

if OCR_SPOTCHECK_CSV_PATH:
    from src.data_processing import load_image

    spotcheck_csv = Path(OCR_SPOTCHECK_CSV_PATH).expanduser()
    if not spotcheck_csv.is_absolute():
        spotcheck_csv = REPO_ROOT / spotcheck_csv
    if not spotcheck_csv.is_file():
        raise FileNotFoundError(f"OCR spot-check CSV not found: {spotcheck_csv}")

    spotcheck_data = pd.read_csv(spotcheck_csv, dtype=str)
    if not {"image_path", "plate_gt"}.issubset(spotcheck_data.columns):
        raise ValueError("The OCR spot-check CSV must include image_path and plate_gt columns.")
    spotcheck_data = spotcheck_data.dropna(subset=["image_path", "plate_gt"])

    spotcheck_reader = RuntimeSelectedSerialReader()
    spotcheck_rows = []
    for row in spotcheck_data.itertuples(index=False):
        image_path = Path(str(row.image_path)).expanduser()
        if not image_path.is_absolute():
            candidates = [spotcheck_csv.parent / image_path, REPO_ROOT / image_path]
            image_path = next((candidate for candidate in candidates if candidate.is_file()), candidates[0])
        if not image_path.is_file():
            print("Skipping a crop missing from the private OCR dataset.")
            continue
        raw, confidence_min, confidence_mean = spotcheck_reader.read(
            load_image(image_path.read_bytes())
        )
        spotcheck_rows.append({
            "image_path": str(image_path),
            "plate_gt_raw": str(row.plate_gt),
            "plate_pred_raw": raw,
            "ocr_confidence_min": confidence_min,
            "ocr_confidence_mean": confidence_mean,
        })
    spotcheck_frame = pd.DataFrame(spotcheck_rows)
    if not spotcheck_frame.empty:
        spotcheck_local_path = LOCAL_PRIVATE_OCR_DIR / "runtime_serial_ocr_spotcheck.csv"
        spotcheck_drive_path = DRIVE_PRIVATE_OCR_DIR / "runtime_serial_ocr_spotcheck.csv"
        spotcheck_frame.to_csv(spotcheck_local_path, index=False)
        persist_file_to_drive(spotcheck_local_path, spotcheck_drive_path)
        print("Raw spot-check rows saved to private Drive:", spotcheck_drive_path)
        display(spotcheck_frame.drop(
            columns=["image_path", "plate_gt_raw", "plate_pred_raw"],
            errors="ignore",
        ).head(20))
    else:
        print("No OCR spot-check rows were available.")
else:
    print("runtime-selected OCR spot-check is disabled. Set OCR_SPOTCHECK_CSV_PATH to inspect a separate crop CSV.")


## 20. Optional end-to-end evaluation on full scenes

This separate evaluation measures detection plus OCR on complete USA scenes. Place the labeled CSV and its scenes in the private Drive evaluation folder; it must contain image_path and plate_gt. Relative scene paths are searched in private Drive, the project clone, then the training dataset. Use a separate scene set that does not overlap training or the detector test. The cell filters detections with the validation-calibrated threshold, crops each candidate, runs OCR, and reports exact match, character error rate, and missed-detection rate. It does not apply access-control rules. Per-image rows with plate references and predictions are copied to private Drive; only an aggregate JSON summary is published to GitHub. The output guard prevents accidental overwriting.

**Confidence intervals.** The aggregate report includes 95% Wilson score intervals for exact-match and missed-detection proportions, plus 95% paired percentile-bootstrap intervals (5,000 resamples of labeled scene images) for exact-match rate and character error rate. Blank-reference rows are excluded before these calculations. The bootstrap sampling unit is the scene image; if images are clustered by vehicle or sequence, a group-level bootstrap would be needed to account for that dependence.

In [ ]:
RUN_SCENE_E2E_TEST = False
SCENE_E2E_REPOSITORY_CSV = REPO_ROOT / "datasets/usa/scene_e2e_test.csv"
SCENE_E2E_CSV = DRIVE_PRIVATE_EVALUATION_DIR / "scene_e2e_test.csv"
if not SCENE_E2E_CSV.is_file() and SCENE_E2E_REPOSITORY_CSV.is_file():
    persist_file_to_drive(SCENE_E2E_REPOSITORY_CSV, SCENE_E2E_CSV)
SCENE_E2E_OUTPUT_PATH = DRIVE_PRIVATE_EVALUATION_DIR / "rtdetr_usa_scene_e2e.csv"
SCENE_E2E_LOCAL_OUTPUT_PATH = LOCAL_PRIVATE_EVALUATION_DIR / "rtdetr_usa_scene_e2e.csv"
SCENE_E2E_SUMMARY_PATH = METRICS_DIR / "rtdetr_usa_scene_e2e_summary.json"

import math
from statistics import NormalDist

def wilson_score_interval(successes, trials, confidence_level=0.95):
    """Wilson score interval for a binomial proportion; bounds are proportions."""
    trials = int(trials)
    successes = int(successes)
    if trials <= 0:
        return {
            "method": "Wilson score",
            "confidence_level": float(confidence_level),
            "lower": None,
            "upper": None,
        }
    z = NormalDist().inv_cdf(0.5 + confidence_level / 2.0)
    p = successes / trials
    z2 = z * z
    denominator = 1.0 + z2 / trials
    center = (p + z2 / (2.0 * trials)) / denominator
    margin = (
        z * math.sqrt(p * (1.0 - p) / trials + z2 / (4.0 * trials * trials))
        / denominator
    )
    return {
        "method": "Wilson score",
        "confidence_level": float(confidence_level),
        "lower": float(max(0.0, center - margin)),
        "upper": float(min(1.0, center + margin)),
    }

def bootstrap_mean_interval(values, confidence_level=0.95, n_resamples=5000, seed=20261110):
    """Percentile bootstrap CI for a mean, resampling the supplied units with replacement."""
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {
            "method": "percentile bootstrap",
            "confidence_level": float(confidence_level),
            "n_resamples": int(n_resamples),
            "lower": None,
            "upper": None,
        }
    rng = np.random.default_rng(seed)
    estimates = np.empty(int(n_resamples), dtype=np.float64)
    batch_size = 256
    for start in range(0, int(n_resamples), batch_size):
        stop = min(start + batch_size, int(n_resamples))
        indices = rng.integers(0, values.size, size=(stop - start, values.size))
        estimates[start:stop] = values[indices].mean(axis=1)
    alpha = 1.0 - confidence_level
    lower, upper = np.quantile(estimates, [alpha / 2.0, 1.0 - alpha / 2.0])
    return {
        "method": "percentile bootstrap",
        "confidence_level": float(confidence_level),
        "n_resamples": int(n_resamples),
        "lower": float(lower),
        "upper": float(upper),
    }

def bootstrap_ratio_interval(
    numerators, denominators, confidence_level=0.95, n_resamples=5000, seed=20261110
):
    """Percentile bootstrap CI for sum(numerator)/sum(denominator), paired by row."""
    numerators = np.asarray(numerators, dtype=np.float64).reshape(-1)
    denominators = np.asarray(denominators, dtype=np.float64).reshape(-1)
    if numerators.size != denominators.size:
        raise ValueError("Bootstrap numerator and denominator arrays must have equal length.")
    valid = (
        np.isfinite(numerators)
        & np.isfinite(denominators)
        & (denominators >= 0)
    )
    numerators = numerators[valid]
    denominators = denominators[valid]
    if numerators.size == 0 or denominators.sum() <= 0:
        return {
            "method": "paired percentile bootstrap",
            "confidence_level": float(confidence_level),
            "n_resamples": int(n_resamples),
            "lower": None,
            "upper": None,
        }
    rng = np.random.default_rng(seed)
    estimates = []
    batch_size = 256
    for start in range(0, int(n_resamples), batch_size):
        batch = min(batch_size, int(n_resamples) - start)
        indices = rng.integers(0, numerators.size, size=(batch, numerators.size))
        numerator_sums = numerators[indices].sum(axis=1)
        denominator_sums = denominators[indices].sum(axis=1)
        usable = denominator_sums > 0
        if usable.any():
            estimates.append(numerator_sums[usable] / denominator_sums[usable])
    if not estimates:
        lower = upper = None
    else:
        values = np.concatenate(estimates)
        alpha = 1.0 - confidence_level
        lower, upper = np.quantile(values, [alpha / 2.0, 1.0 - alpha / 2.0])
        lower, upper = float(lower), float(upper)
    return {
        "method": "paired percentile bootstrap",
        "confidence_level": float(confidence_level),
        "n_resamples": int(n_resamples),
        "lower": lower,
        "upper": upper,
    }

def summarize_scene_confidence_intervals(
    frame, exact_match_col, detected_col, char_errors_col, reference_col, seed
):
    """Calculate Wilson and image-level bootstrap intervals for end-to-end outcomes."""
    exact = frame[exact_match_col].fillna(False).astype(bool).to_numpy(dtype=np.float64)
    detected = frame[detected_col].fillna(False).astype(bool).to_numpy(dtype=bool)
    char_errors = pd.to_numeric(frame[char_errors_col], errors="coerce").to_numpy(dtype=np.float64)
    reference_lengths = (
        frame[reference_col].fillna("").astype(str).str.len().to_numpy(dtype=np.float64)
    )
    n = int(exact.size)
    missed = (~detected).astype(np.float64)
    valid_chars = (
        np.isfinite(char_errors)
        & np.isfinite(reference_lengths)
        & (reference_lengths > 0)
    )
    errors = char_errors[valid_chars]
    lengths = reference_lengths[valid_chars]
    character_error_rate = (
        float(errors.sum() / lengths.sum()) if lengths.size and lengths.sum() > 0 else None
    )
    return {
        "confidence_level": 0.95,
        "sampling_unit": "labeled scene image",
        "exact_match_rate": {
            "n": n,
            "successes": int(exact.sum()),
            "point_estimate": float(exact.mean()) if n else None,
            "wilson_score": wilson_score_interval(int(exact.sum()), n),
            "bootstrap_percentile": bootstrap_mean_interval(
                exact, n_resamples=5000, seed=seed
            ),
        },
        "missed_detection_rate": {
            "n": n,
            "misses": int(missed.sum()),
            "point_estimate": float(missed.mean()) if n else None,
            "wilson_score": wilson_score_interval(int(missed.sum()), n),
            "bootstrap_percentile": bootstrap_mean_interval(
                missed, n_resamples=5000, seed=seed + 1
            ),
        },
        "character_error_rate": {
            "n_images": int(lengths.size),
            "point_estimate": character_error_rate,
            "bootstrap_percentile": bootstrap_ratio_interval(
                errors, lengths, n_resamples=5000, seed=seed + 2
            ),
        },
    }

if RUN_SCENE_E2E_TEST:
    from src.data_processing import clean_text, crop_with_margin, plate_distance

    if not COUNTRY_IR_XML.is_file():
        raise RuntimeError("The USA fine-tuned IR is missing.")
    if not MODEL_CHECKPOINT_FROZEN:
        raise RuntimeError("Freeze the selected checkpoint before running the scene-level evaluation.")
    if SCENE_E2E_OUTPUT_PATH.exists():
        raise FileExistsError(f"Scene-level output already exists: {SCENE_E2E_OUTPUT_PATH}")
    threshold_file = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_file.is_file():
        raise RuntimeError("Calibrate the USA confidence threshold on validation first.")
    threshold_record = json.loads(threshold_file.read_text(encoding="utf-8"))
    if (
        threshold_record.get("checkpoint") != str(COUNTRY_IR_DIR)
        or threshold_record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    if not threshold_record.get("usable_for_inference", False) or threshold_record.get("threshold") is None:
        raise RuntimeError("Validation did not meet the operational target; threshold-based inference and evaluation are blocked.")
    ACTIVE_THRESHOLD = float(threshold_record["threshold"])
    if not SCENE_E2E_CSV.is_file():
        raise FileNotFoundError("Create scene_e2e_test.csv with image_path and plate_gt columns.")
    scene_truth = pd.read_csv(SCENE_E2E_CSV, dtype=str)
    if not {"image_path", "plate_gt"}.issubset(scene_truth.columns):
        raise ValueError("The CSV must contain image_path and plate_gt columns.")
    scene_truth = scene_truth.dropna(subset=["image_path", "plate_gt"])
    recognizer = RuntimeSelectedSerialReader()
    print("Registration-serial OCR backend:", recognizer.runtime_name)
    predictions = []
    for row in scene_truth.itertuples(index=False):
        candidate_path = Path(row.image_path)
        if not candidate_path.is_absolute():
            options = [
                DRIVE_PRIVATE_EVALUATION_DIR / candidate_path,
                REPO_ROOT / candidate_path,
                Path(DATA_DIR) / candidate_path,
            ]
            candidate_path = next((p for p in options if p.is_file()), options[0])
        image = load_image(candidate_path.read_bytes())
        detections = [
            d for d in detector.detect(image, max_detections=20)
            if d.confidence >= ACTIVE_THRESHOLD
        ]
        candidates = []
        for detection in detections:
            crop = crop_with_margin(image, detection.box, 0.06)
            if crop.size == 0:
                continue
            read = recognizer.run(crop, return_confidence=True)[0]
            text = clean_text(read.plate)
            probs = np.asarray(read.char_probs, dtype=np.float32).ravel()
            confidence = float(probs.mean()) if probs.size else 0.0
            candidates.append((text, confidence, detection.confidence))
        best = max(candidates, key=lambda item: (item[1], item[2])) if candidates else ("", 0.0, 0.0)
        gt = clean_text(row.plate_gt)
        predictions.append({
            "image_path": str(candidate_path),
            "plate_gt": gt,
            "plate_pred": best[0],
            "exact_match": best[0] == gt,
            "char_errors": plate_distance(best[0], gt),
            "detected": bool(detections),
            "det_confidence": best[2],
            "ocr_confidence": best[1],
        })
    e2e_results = pd.DataFrame(predictions)
    scene_e2e_summary = {
        "n": len(e2e_results),
        "exact_match_rate": float(e2e_results["exact_match"].mean()) if len(e2e_results) else 0.0,
        "character_error_rate": float(
            e2e_results["char_errors"].sum() / max(e2e_results["plate_gt"].str.len().sum(), 1)
        ) if len(e2e_results) else 0.0,
        "missed_detection_rate": float((~e2e_results["detected"]).mean()) if len(e2e_results) else 0.0,
    }
    scene_e2e_summary["confidence_intervals_95"] = summarize_scene_confidence_intervals(
        e2e_results,
        exact_match_col="exact_match",
        detected_col="detected",
        char_errors_col="char_errors",
        reference_col="plate_gt",
        seed=20261120,
    )

    LOCAL_PRIVATE_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
    e2e_results.to_csv(SCENE_E2E_LOCAL_OUTPUT_PATH, index=False)
    persist_file_to_drive(SCENE_E2E_LOCAL_OUTPUT_PATH, SCENE_E2E_OUTPUT_PATH)
    SCENE_E2E_SUMMARY_PATH.write_text(
        json.dumps(scene_e2e_summary, indent=2), encoding="utf-8"
    )
    publish_small_metrics(
        [SCENE_E2E_SUMMARY_PATH],
        "Publish USA full-scene evaluation summary",
    )
    print("Aggregate full-scene metrics:", scene_e2e_summary)
    print("Per-image OCR rows saved to private Drive:", SCENE_E2E_OUTPUT_PATH)
else:
    print("End-to-end test is disabled.")

## 21. Optional end-to-end evaluation with the SIRAP pipeline

This evaluation runs the actual SIRAP ALPRPipeline on a separately labeled set of complete scenes. It measures the detector and the selected serial OCR reader together, unlike the crop-only OCR section. The notebook constructs the SIRAP pipeline with its CPU-aware ONNX serial reader. Intel CPUs use OpenVINOExecutionProvider first; other CPUs use CPUExecutionProvider. The code does not instantiate a second OCR session. The labeled CSV is read from the private Drive evaluation folder and must contain image_path and plate_gt. Relative scene paths are searched in private Drive, the project clone, then the training dataset. The cell applies the validation-calibrated confidence threshold, copies per-image rows containing plate text to private Drive, and publishes only the aggregate metrics JSON to GitHub. It refuses to overwrite an existing report. Use a scene set that is disjoint from training and the detector holdout test.

**Confidence intervals.** This report uses the same two methods on the SIRAP pipeline's per-image results: Wilson score intervals for exact-match and missed-detection proportions, and paired percentile-bootstrap intervals for exact-match rate and character error rate.

In [ ]:
RUN_SIRAP_PIPELINE_E2E = False
SIRAP_PIPELINE_E2E_CSV = SCENE_E2E_CSV
SIRAP_PIPELINE_E2E_CSV_OUT = DRIVE_PRIVATE_EVALUATION_DIR / "rtdetr_usa_sirap_pipeline_e2e.csv"
SIRAP_PIPELINE_E2E_LOCAL_CSV_OUT = LOCAL_PRIVATE_EVALUATION_DIR / "rtdetr_usa_sirap_pipeline_e2e.csv"
SIRAP_PIPELINE_E2E_JSON_OUT = METRICS_DIR / "rtdetr_usa_sirap_pipeline_e2e.json"

if RUN_SIRAP_PIPELINE_E2E:
    if not MODEL_CHECKPOINT_FROZEN or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Freeze the selected USA checkpoint before running pipeline evaluation.")
    if not SIRAP_PIPELINE_E2E_CSV.is_file():
        raise FileNotFoundError("The labeled full-scene CSV is missing.")
    threshold_file = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_file.is_file():
        raise RuntimeError("Calibrate the USA confidence threshold on validation first.")
    threshold_record = json.loads(threshold_file.read_text(encoding="utf-8"))
    if (
        threshold_record.get("checkpoint") != str(COUNTRY_IR_DIR)
        or threshold_record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    if not threshold_record.get("usable_for_inference", False) or threshold_record.get("threshold") is None:
        raise RuntimeError("Validation did not meet the operational target; threshold-based inference and evaluation are blocked.")
    ACTIVE_THRESHOLD = float(threshold_record["threshold"])
    if SIRAP_PIPELINE_E2E_CSV_OUT.exists() or SIRAP_PIPELINE_E2E_JSON_OUT.exists():
        raise FileExistsError("A SIRAP pipeline end-to-end report already exists.")
    scene_data = pd.read_csv(SIRAP_PIPELINE_E2E_CSV, dtype=str)
    if not {"image_path", "plate_gt"}.issubset(scene_data.columns):
        raise ValueError("The CSV must contain image_path and plate_gt columns.")

    import yaml
    from src.model import ALPRPipeline, PlateDetector
    from src.data_processing import clean_text, load_image, plate_distance

    usa_config = yaml.safe_load((REPO_ROOT / "config/countries/usa.yaml").read_text(encoding="utf-8"))
    detector_config = usa_config["detector"]
    pipeline_detector = PlateDetector(
        checkpoint=DETECTOR_DIR,
        model_id=detector_config.get("model_id", MODEL_ID),
        imgsz=detector_config.get("inference_size", IMAGE_SIZE),
        conf=ACTIVE_THRESHOLD,
        class_id=detector_config.get("class_id", 0),
        device=detector_config.get("device", "CPU"),
        fallback_checkpoint=detector_config.get("fallback_checkpoint"),
    )
    pipeline = ALPRPipeline(
        detector=pipeline_detector,
        reader=RuntimeSelectedSerialReader(),
        margin=detector_config.get("crop_margin", 0.06),
    )
    print("Registration-serial OCR backend:", pipeline.reader.runtime_name)
    rows = []
    for row in scene_data.dropna(subset=["image_path", "plate_gt"]).itertuples(index=False):
        image_path = Path(row.image_path)
        if not image_path.is_absolute():
            candidates = [
                DRIVE_PRIVATE_EVALUATION_DIR / image_path,
                REPO_ROOT / image_path,
                Path(DATA_DIR) / image_path,
            ]
            image_path = next((candidate for candidate in candidates if candidate.is_file()), candidates[0])
        recognition = pipeline.recognize(load_image(image_path.read_bytes()))
        reference = clean_text(row.plate_gt)
        prediction = clean_text(recognition.plate or "")
        rows.append({
            "image_path": str(image_path),
            "plate_reference": reference,
            "plate_prediction": prediction,
            "exact_match": prediction == reference,
            "character_errors": plate_distance(prediction, reference),
            "detector_confidence": recognition.det_confidence,
            "detected": recognition.box is not None,
            "ocr_confidence": recognition.ocr_confidence,
            "latency_ms": recognition.timings_ms.get("total"),
        })

    pipeline_results = pd.DataFrame(rows)
    result_summary = {
        "detector": pipeline.detector.name,
        "checkpoint": pipeline.detector.source,
        "image_count": len(pipeline_results),
        "exact_match_rate": float(pipeline_results["exact_match"].mean()) if len(pipeline_results) else 0.0,
        "character_error_rate": float(
            pipeline_results["character_errors"].sum()
            / max(pipeline_results["plate_reference"].str.len().sum(), 1)
        ) if len(pipeline_results) else 0.0,
        "missed_detection_rate": float((~pipeline_results["detected"]).mean())
        if len(pipeline_results) else 0.0,
        "operational_confidence_threshold": ACTIVE_THRESHOLD,
    }
    result_summary["confidence_intervals_95"] = summarize_scene_confidence_intervals(
        pipeline_results,
        exact_match_col="exact_match",
        detected_col="detected",
        char_errors_col="character_errors",
        reference_col="plate_reference",
        seed=20261121,
    )

    LOCAL_PRIVATE_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
    pipeline_results.to_csv(SIRAP_PIPELINE_E2E_LOCAL_CSV_OUT, index=False)
    persist_file_to_drive(SIRAP_PIPELINE_E2E_LOCAL_CSV_OUT, SIRAP_PIPELINE_E2E_CSV_OUT)
    SIRAP_PIPELINE_E2E_JSON_OUT.write_text(json.dumps(result_summary, indent=2), encoding="utf-8")
    publish_small_metrics(
        [SIRAP_PIPELINE_E2E_JSON_OUT],
        "Publish USA SIRAP pipeline evaluation summary",
    )
    print("Aggregate SIRAP pipeline metrics:", result_summary)
    print("Per-image OCR rows saved to private Drive:", SIRAP_PIPELINE_E2E_CSV_OUT)
else:
    print("SIRAP pipeline end-to-end evaluation is disabled.")

## 22. Evaluate the held-out detector test exactly once

This is the final COCO detector evaluation. It requires the grouped split audit, a written model-selection record, the country-specific OpenVINO IR, and at least 400 held-out test images. Before running it, set MODEL_CHECKPOINT_FROZEN to True and fill MODEL_SELECTION_RECORD with the selected checkpoint, validation metric, code revision, and date. The cell refuses to overwrite an existing final metrics file.

The report records the split seed, test size, group count, plate count, grouping limitations, model-selection record, operational threshold, and COCO scoring threshold. Do not use this test output to revise the model or threshold.

The final report also includes Wilson score intervals and image-cluster percentile-bootstrap intervals for precision and recall at the validation-calibrated operating threshold (IoU ≥ 0.50). COCO AP/mAP remains a point estimate here; Wilson intervals do not apply to AP.

The repository already contains a legacy `rtdetr_usa_independent_test.json` whose model-selection record identifies a different seed-0 candidate. Do not use it as the final result for the selected seed-1 checkpoint. Preserve the selected checkpoint's test set until the split and calibration meet the documented requirements.

In [ ]:
from pathlib import Path
import json
from src.evaluate import evaluate_detector

RUN_FINAL_TEST_EVALUATION = False
FINAL_OPENVINO_CHECKPOINT = MODEL_OUTPUT / "openvino"
FINAL_TEST_METRICS_PATH = REPO_ROOT / "results/metrics/rtdetr_usa_independent_test.json"

if RUN_FINAL_TEST_EVALUATION:
    if not globals().get("INDEPENDENT_TEST_READY", False):
        raise RuntimeError("Create and audit the grouped holdout split first.")
    if not MODEL_CHECKPOINT_FROZEN or not MODEL_SELECTION_RECORD.strip():
        raise RuntimeError("Record and freeze the selected model using validation only.")
    if not FINAL_OPENVINO_CHECKPOINT.is_dir():
        raise FileNotFoundError(f"OpenVINO checkpoint does not exist: {FINAL_OPENVINO_CHECKPOINT}")
    threshold_file = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_file.is_file():
        raise RuntimeError("Calibrate the USA confidence threshold on validation first.")
    threshold_record = json.loads(threshold_file.read_text(encoding="utf-8"))
    if (
        threshold_record.get("checkpoint") != str(FINAL_OPENVINO_CHECKPOINT)
        or threshold_record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    if not threshold_record.get("usable_for_inference", False) or threshold_record.get("threshold") is None:
        raise RuntimeError("Validation did not meet the operational target; threshold-based inference and evaluation are blocked.")
    ACTIVE_THRESHOLD = float(threshold_record["threshold"])
    if FINAL_TEST_METRICS_PATH.exists():
        raise FileExistsError(
            f"{FINAL_TEST_METRICS_PATH} already exists; the final test will not be evaluated again."
        )
    if SPLIT_SUMMARY["splits"]["test"]["images"] < MINIMUM_TEST_IMAGES:
        raise RuntimeError("The test split is smaller than the configured minimum.")

    FINAL_TEST_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_metrics_path = FINAL_TEST_METRICS_PATH.with_suffix(".working.json")
    if temporary_metrics_path.exists():
        temporary_metrics_path.unlink()

    final_test_metrics = evaluate_detector(
        data_dir=DATA_DIR,
        country="usa",
        split="test",
        checkpoint=FINAL_OPENVINO_CHECKPOINT,
        model_id=MODEL_ID,
        threshold=0.001,
        result_path=temporary_metrics_path,
    )
    final_test_metrics["test_protocol"] = {
        "source_dataset": SPLIT_SUMMARY["source_dataset"],
        "seed": SPLIT_SEED,
        "test_design": SPLIT_SUMMARY["test_design"],
        "test_images": SPLIT_SUMMARY["splits"]["test"]["images"],
        "test_groups": SPLIT_SUMMARY["splits"]["test"]["groups"],
        "test_plates": SPLIT_SUMMARY["splits"]["test"]["plates"],
        "grouping_level": SPLIT_SUMMARY["grouping_level"],
        "model_selection_record": MODEL_SELECTION_RECORD,
        "threshold_for_ap": 0.001,
        "operational_threshold": ACTIVE_THRESHOLD,
    }
    test_records_for_ci = collect_detection_records("test")
    per_image_test_counts = [
        score_detection_records([record], ACTIVE_THRESHOLD, iou_threshold=0.50)
        for record in test_records_for_ci
    ]
    tp_by_image = np.asarray([item["tp"] for item in per_image_test_counts], dtype=np.float64)
    fp_by_image = np.asarray([item["fp"] for item in per_image_test_counts], dtype=np.float64)
    fn_by_image = np.asarray([item["fn"] for item in per_image_test_counts], dtype=np.float64)
    test_tp = int(tp_by_image.sum())
    test_fp = int(fp_by_image.sum())
    test_fn = int(fn_by_image.sum())
    precision_trials = test_tp + test_fp
    recall_trials = test_tp + test_fn
    final_test_metrics["operational_threshold_detection_metrics"] = {
        "split": "test",
        "image_count": len(test_records_for_ci),
        "iou_threshold": 0.50,
        "operating_confidence_threshold": ACTIVE_THRESHOLD,
        "matching_method": "greedy one-to-one IoU matching",
        "counts": {
            "true_positives": test_tp,
            "false_positives": test_fp,
            "false_negatives": test_fn,
        },
        "precision": {
            "point_estimate": test_tp / precision_trials if precision_trials else None,
            "wilson_score_95": wilson_score_interval(test_tp, precision_trials),
            "image_cluster_bootstrap_percentile_95": bootstrap_ratio_interval(
                tp_by_image,
                tp_by_image + fp_by_image,
                n_resamples=5000,
                seed=20261122,
            ),
        },
        "recall": {
            "point_estimate": test_tp / recall_trials if recall_trials else None,
            "wilson_score_95": wilson_score_interval(test_tp, recall_trials),
            "image_cluster_bootstrap_percentile_95": bootstrap_ratio_interval(
                tp_by_image,
                tp_by_image + fn_by_image,
                n_resamples=5000,
                seed=20261123,
            ),
        },
    }

    FINAL_TEST_METRICS_PATH.write_text(
        json.dumps(final_test_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    temporary_metrics_path.unlink(missing_ok=True)
    publish_small_metrics(
        [FINAL_TEST_METRICS_PATH],
        "Publish USA independent detector test metrics",
    )
    print(json.dumps(final_test_metrics, indent=2, ensure_ascii=False))
else:
    print(
        "The final test has not been evaluated. Train and select using train/validation; "
        "freeze the model and then set RUN_FINAL_TEST_EVALUATION to True exactly once."
    )

## 23. Optional J.D. Power vehicle-spec lookup by plate

This section looks up vehicle-level details using a U.S. plate and state. It can use the two demonstration pairs supplied below or read a small, capped set of state/serial pairs from section 19's OCR CSV. The detailed lookup table is saved to private Drive; the notebook displays only status counts. It is not a detector/OCR evaluation metric.

The lookup uses J.D. Power's public [License Plate Lookup & Decoder](https://www.jdpower.com/cars/license-plate-lookup-and-decoder) report route. The page says its lookup returns vehicle information, not the owner's name or address. Use only plates you are authorized to process. The browser uses standard Playwright; if the site returns an access-denied or rate-limit response, the code records it and does not retry or try to bypass it. Detailed plate and vehicle fields are stored only in the private Drive folder.

Set the browser setup switch to True and run subsection 23.1 once. Then run subsection 23.2. For lookup pairs from section 19, set \`JDPOWER_INPUT_SOURCE = "section19_csv"\`; by default, only the two demonstration plates are queried.

### 23.1 Install the optional browser and plotting dependencies

This cell installs Playwright and its Chromium browser for the optional lookup, plus the table/plot libraries used by the result display. It does not run any lookup.

In [ ]:
import subprocess
import sys

INSTALL_JDPOWER_BROWSER = True

if INSTALL_JDPOWER_BROWSER:
    lookup_packages = [
        "camoufox[geoip]",
        "pandas>=2.2,<4",
        "matplotlib>=3.8",
        "seaborn>=0.13,<1",
    ]
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *lookup_packages
    ])
    subprocess.check_call([
        sys.executable, "-m", "camoufox", "fetch"
    ])
    print("Camoufox (anti-detect) and dependencies are ready for section 23.")
else:
    print("Browser setup was skipped. Set INSTALL_JDPOWER_BROWSER = True to install it.")

### 23.2 Query the selected plate/state pairs

This cell uses a low concurrency limit, spaces out request starts, saves the HTTP status and source URL, and extracts a small set of visible vehicle specifications. It stops short of retrying access-denied or rate-limited requests. The optional section-19 input is capped to prevent a whole OCR dataset from becoming a large lookup batch.

The Colab/Jupyter cell uses top-level \`await\` because the notebook already has an active event loop; do not wrap it in \`asyncio.run()\`.

In [ ]:
import asyncio
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from camoufox.async_api import AsyncCamoufox

CONCURRENCY_LIMIT = 2
MIN_SECONDS_BETWEEN_REQUESTS = 2.0
PAGE_TIMEOUT_MS = 30_000
MAX_OCR_LOOKUPS = 5

# "demo" uses the two sample pairs supplied for this notebook.
# "section19_csv" uses jurisdiction_abbreviation_pred + registration_serial_pred.
JDPOWER_INPUT_SOURCE = "demo"
JDPOWER_DEMO_PLATES = [("6ZGY984", "CA"), ("CX9S159", "TX")]
JDPOWER_OCR_FIELDS_CSV = METRICS_DIR / "rtdetr_usa_plate_fields_ocr.csv"

# Run only this optional lookup section when you intend to query J.D. Power.
RUN_JDPOWER_LOOKUP = False
SAVE_JDPOWER_RESULTS = False
JDPOWER_LOCAL_OUTPUT = Path("/content/SIRAP_workspace/vehicle_lookup/jdpower_plate_specs.csv")

JDPOWER_REPORT_BASE = (
    "https://www.jdpower.com/cars/license-plate-lookup-and-decoder"
)
BLOCKED_ASSETS = {"image", "font", "media"}
BLOCKED_TRACKERS = (
    "google-analytics.com",
    "googletagmanager.com",
    "doubleclick.net",
)
SPEC_LABELS = {
    "exterior": "exterior",
    "interior": "interior",
    "mpg": "mpg",
    "horsepower": "horsepower",
    "fuel type": "fuel_type",
    "transmission": "transmission",
    "drivetrain": "drivetrain",
    "displacement": "displacement",
    "cylinders": "cylinders",
    "fuel tank capacity": "fuel_tank_capacity",
    "towing capacity": "towing_capacity",
    "seating capacity": "seating_capacity",
}

def build_jdpower_workload():
    """Return validated (serial, state-code) pairs for the bounded lookup."""
    if JDPOWER_INPUT_SOURCE == "demo":
        rows = JDPOWER_DEMO_PLATES
    elif JDPOWER_INPUT_SOURCE == "section19_csv":
        if not JDPOWER_OCR_FIELDS_CSV.is_file():
            raise FileNotFoundError(
                "Section 19 OCR output is missing. Run its separate-field OCR cell first."
            )
        frame = pd.read_csv(JDPOWER_OCR_FIELDS_CSV, dtype=str).fillna("")
        state_column = (
            "jurisdiction_abbreviation_pred"
            if "jurisdiction_abbreviation_pred" in frame.columns
            else "state_abbreviation_pred"
        )
        serial_column = "registration_serial_pred"
        if state_column not in frame.columns or serial_column not in frame.columns:
            raise ValueError(
                "Section 19 CSV must contain jurisdiction_abbreviation_pred "
                "(or state_abbreviation_pred) and registration_serial_pred."
            )
        rows = list(zip(frame[serial_column], frame[state_column]))
    else:
        raise ValueError("JDPOWER_INPUT_SOURCE must be 'demo' or 'section19_csv'.")

    cleaned = []
    seen = set()
    for serial, state in rows:
        serial = re.sub(r"[^A-Z0-9]", "", str(serial).upper())
        state = re.sub(r"[^A-Z]", "", str(state).upper())
        if not serial or len(state) != 2:
            continue
        key = (serial, state)
        if key not in seen:
            cleaned.append(key)
            seen.add(key)
        if JDPOWER_INPUT_SOURCE == "section19_csv" and len(cleaned) >= MAX_OCR_LOOKUPS:
            break
    return cleaned

def parse_visible_specs(report_text):
    """Extract a limited set of adjacent label/value pairs from the visible report text."""
    lines = [" ".join(line.split()) for line in report_text.splitlines()]
    lines = [line for line in lines if line]
    specs = {}
    labels = {label.casefold(): key for label, key in SPEC_LABELS.items()}

    for index, line in enumerate(lines):
        folded = line.casefold().strip().rstrip(":")
        if folded in labels:
            if index + 1 < len(lines):
                value = lines[index + 1].strip()
                if value and value.casefold() not in labels:
                    specs[labels[folded]] = value
            continue

        for label, key in SPEC_LABELS.items():
            prefix = label.casefold() + ":"
            if line.casefold().startswith(prefix):
                value = line[len(label) + 1:].strip()
                if value:
                    specs[key] = value
    return specs

async def intercept_static_assets(route):
    request = route.request
    url = request.url.lower()
    is_tracker = any(domain in url for domain in BLOCKED_TRACKERS)
    if request.resource_type in BLOCKED_ASSETS or is_tracker:
        await route.abort()
    else:
        await route.continue_()

async def scrape_single_plate(browser, plate, state, semaphore, request_schedule, schedule_lock):
    plate = re.sub(r"[^A-Z0-9]", "", str(plate).upper())
    state = re.sub(r"[^A-Z]", "", str(state).upper())
    record = {
        "plate": plate,
        "state": state,
        "lookup_key": f"{state}:{plate}" if state and plate else "",
        "lookup_url": "",
        "lookup_status": "not_started",
        "vehicle_title": "",
        "specs": {},
        "error": "",
    }
    if not plate or len(state) != 2:
        record["lookup_status"] = "invalid_input"
        record["error"] = "Plate must be alphanumeric and state must be a two-letter code."
        return record

    record["lookup_url"] = (
        f"{JDPOWER_REPORT_BASE}/{plate.lower()}-{state.lower()}-license-plate-report"
    )
    async with semaphore:
        loop = asyncio.get_running_loop()
        async with schedule_lock:
            delay = max(0.0, request_schedule["next_start"] - loop.time())
            if delay:
                await asyncio.sleep(delay)
            request_schedule["next_start"] = loop.time() + MIN_SECONDS_BETWEEN_REQUESTS

        page = await browser.new_page()
        page.set_default_navigation_timeout(PAGE_TIMEOUT_MS)
        await page.route("**/*", intercept_static_assets)
        try:
            print(f"[Section 23] Looking up {plate} ({state})")
            response = await page.goto(
                record["lookup_url"],
                wait_until="domcontentloaded",
                timeout=PAGE_TIMEOUT_MS,
            )
            if response is None:
                record["lookup_status"] = "no_response"
                record["error"] = "The browser received no HTTP response."
                return record

            status = response.status
            if status in (401, 403, 429):
                record["lookup_status"] = f"access_limited_{status}"
                record["error"] = (
                    f"J.D. Power returned HTTP {status}; no retry was attempted."
                )
                print("   Lookup stopped for this plate:", record["error"])
                return record
            if status != 200:
                record["lookup_status"] = f"http_{status}"
                record["error"] = f"J.D. Power returned HTTP {status}."
                return record

            record["lookup_status"] = "http_200"
            heading = page.locator("h1").first
            await heading.wait_for(state="visible", timeout=10_000)
            record["vehicle_title"] = (
                (await heading.text_content() or "").strip()
            )

            main = page.locator("main")
            report_text = (
                await main.inner_text(timeout=5_000)
                if await main.count()
                else await page.locator("body").inner_text(timeout=5_000)
            )
            record["specs"] = parse_visible_specs(report_text)
            record["lookup_status"] = (
                "found" if record["vehicle_title"] else "no_report_title"
            )
        except Exception as error:
            record["lookup_status"] = "error"
            record["error"] = f"{type(error).__name__}: {error}"
            print(f"   Lookup issue for {plate} ({state}): {record['error']}")
        finally:
            await page.close()
    return record

async def execute_sirap_jdpower_job(workload):
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    request_schedule = {"next_start": 0.0}
    schedule_lock = asyncio.Lock()

    # Se reemplaza Playwright puro con AsyncCamoufox (headless=True)
    async with AsyncCamoufox(headless=True) as browser:
        tasks = [
            scrape_single_plate(
                browser, plate, state, semaphore, request_schedule, schedule_lock
            )
            for plate, state in workload
        ]
        return await asyncio.gather(*tasks)

if RUN_JDPOWER_LOOKUP:
    lookup_workload = build_jdpower_workload()
    if not lookup_workload:
        raise RuntimeError("No valid state/serial pairs were found for section 23.")

    jdpower_records = await execute_sirap_jdpower_job(lookup_workload)
    df_sirap_specs = pd.json_normalize(jdpower_records, sep="_")
    display(df_sirap_specs)

    if SAVE_JDPOWER_RESULTS:
        JDPOWER_LOCAL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        df_sirap_specs.to_csv(JDPOWER_LOCAL_OUTPUT, index=False)
        print("Local lookup file saved to:", JDPOWER_LOCAL_OUTPUT)

    if not df_sirap_specs.empty and "lookup_status" in df_sirap_specs:
        plt.figure(figsize=(8, 4))
        sns.countplot(data=df_sirap_specs, x="lookup_status")
        plt.title("J.D. Power lookup outcomes")
        plt.xlabel("Lookup status")
        plt.ylabel("Number of plates")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("J.D. Power lookup is disabled. Set RUN_JDPOWER_LOOKUP = True to run it.")


## 24. Pre-deployment checklist

Before using the model in access decisions, confirm that:

- The USA data covers different plate designs and states, day/night conditions, glare, weather, angles, distances, small plates, and occlusions.
- Related images are grouped by vehicle, camera, session, or sequence when those identifiers exist. Inspect the saved split manifest for overlap and exact duplicates.
- The detector report includes mAP50-95, AP50, AP75, precision, recall, F1, false positives, false negatives, and small-object results.
- The confidence threshold was selected on validation according to the costs of false acceptances and false rejections.
- Robustness diagnostics include real scenes as well as synthetic image perturbations.
- Crop OCR and full-scene end-to-end metrics are reported separately, including exact-match rate, character error rate, and missed detections.
- CPU latency uses a documented hardware class and the same input size as deployment.
- Any future automated gate is assessed with a separate labeled decision dataset; uncertain readings are sent for human review.

A high detector score alone does not establish a safe access-control system. The starting checkpoint's published metrics describe its original evaluation set and are not USA SIRAP results.

- If the calibration report has `usable_for_inference=false` or `threshold=null`, do not run live or access-control inference.
- Keep the 30 PKLot images in validation only; add new, separately reviewed target-camera negatives to training and validation before retraining.

## 25. Optionally publish the reviewed OpenVINO IR

Publishing is disabled by default. The selected model and full Transformers checkpoint remain in Drive. After reviewing results, set PUBLISH_IR_TO_SPACE to True to stage only the OpenVINO deployment bundle from Drive under models/usa/detector/rtdetrv2_s/openvino/ in the repository. The helper requires GITHUB_TOKEN in Colab Secrets with repository Contents write access; repository automation can then synchronize the deployment IR with the Hugging Face Space. Dataset archives and raw row-level OCR/lookups are not published.

In [ ]:
PUBLISH_IR_TO_SPACE = False
if PUBLISH_IR_TO_SPACE:
    from scripts.notebook_io import publish_detector_ir
    if not COUNTRY_IR_XML.is_file():
        raise FileNotFoundError("Train and export the USA model first.")
    if REPO_MODEL_OUTPUT.exists():
        shutil.rmtree(REPO_MODEL_OUTPUT)
    REPO_MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(COUNTRY_IR_DIR, REPO_MODEL_OUTPUT / "openvino")
    print("Staged the selected OpenVINO deployment IR from Drive:", REPO_MODEL_OUTPUT / "openvino")
    print(publish_detector_ir(REPO_ROOT, COUNTRY))
else:
    print("Publishing is disabled.")


## 26. Result files and reporting

Training/evaluation cells create results only when their run switches are enabled and required inputs are available. Do not report metric values before the matching cell has finished.

**GitHub (results/metrics/).** Compact aggregate outputs are published after their producing stage when the Colab Secret GITHUB_TOKEN is set: Optuna summaries, three-seed validation, the selected training manifest, split counts, validation threshold calibration, robustness and latency summaries, OCR accuracy summaries, full-scene/pipeline aggregate scores, and the one-time independent detector test JSON.

**Google Drive (MyDrive/SIRAP_USA_ECUADOR/).** The source COCO dataset and generated independent split are stored as .tar archives under datasets/usa/archives/. Section 4 restores an archive into local /content for training and evaluation. The selected Transformers checkpoint, OpenVINO IR, and selected run archive are stored under models/usa/ and checkpoints/usa/. Large video outputs and generated camera scripts are also copied to Drive.

**Private Drive (private/usa/).** Per-image OCR CSVs, full-scene and SIRAP pipeline rows, J.D. Power lookup tables, camera readings, and annotated camera videos are copied here. Keep this folder unshared. The notebook displays only aggregate lookup counts and redacted OCR/session previews, and these row-level files are excluded from GitHub commits.

The optional section 25 publisher stages the selected OpenVINO IR in the repository only when deploying it to the Hugging Face Space. GitHub's model ignore rules and the publisher's file-size checks still apply. A Hugging Face Space does not automatically mount personal Drive; this notebook's Drive cache is for Colab.

The imported training helper also writes its training manifest. Each report should identify dataset/version, split sizes, group counts, checkpoint, confidence threshold, runtime, and whether a metric came from validation or the final test.